In [ ]:
!python -c "import monai" || pip install -q "monai-weekly[tqdm, nibabel, gdown, ignite, matplotlib]"

In [ ]:
pip install monai

In [ ]:
import monai.transforms as mt
import matplotlib.pyplot as plt
import ignite
import numpy as np
import torch
import monai
import warnings

warnings.filterwarnings("ignore")  # remove some scikit-image warnings

monai.config.print_config()

Pretraining

In [ ]:
!pip install -q "monai[all]" torch torchvision nibabel tqdm matplotlib

Initial Dependacies

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from monai.data import Dataset, CacheDataset, DataLoader
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, NormalizeIntensityd, RandSpatialCropd,
    RandFlipd, RandRotate90d, Compose, ScaleIntensityRanged
)
from monai.networks.nets import SwinUNETR
from monai.losses import SSIMLoss
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Data Preprocessing

In [ ]:
data_dir = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/volumes"
volume_files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".nii") or f.endswith(".nii.gz")])

data_dicts = [{"image": f} for f in volume_files]
print(f"Found {len(data_dicts)} volumes.")
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
    RandSpatialCropd(keys=["image"], roi_size=(96, 96, 96), random_size=False),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image"], prob=0.5, spatial_axes=(0, 1)),
])

train_ds = CacheDataset(data=data_dicts, transform=train_transforms, cache_rate=1.0)
train_loader = DataLoader(train_ds, batch_size=2)
print(f"Train dataset size: {len(train_ds)}")

In [ ]:

for batch in train_loader:
    img = batch["image"]
    print("Batch shape:", img.shape)  # (B, C, H, W, D)
    break

3D Volume Dimensions

In [ ]:
import nibabel as nib

path = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/volumes/UCSF-PDGM-0004_T2.nii.gz"
img = nib.load(path)
data = img.get_fdata()

print("Shape:", data.shape)
print("Voxel size:", img.header.get_zooms())
print("Orientation:", nib.aff2axcodes(img.affine))

In [ ]:
print(sum(p.numel() for p in encoder.parameters()) / 1e6, "M parameters")

Masking functions

In [ ]:
a = torch.randn(4, 128)
b = torch.randn(4, 128)
loss = contrastive_loss(a, b)
print(loss)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR


class MAEContrastivePretrainer(nn.Module):
    def __init__(self, feature_size=48, proj_dim=128, mask_ratio=0.75):
        super().__init__()
        self.encoder = SwinUNETR(in_channels=1, out_channels=feature_size, feature_size=feature_size)
        self.mask_ratio = mask_ratio

        # MAE decoder (reconstruct masked voxels)
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(768, 384, 2, stride=2),
            nn.ReLU(),
            nn.Conv3d(384, 192, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(192, 1, 3, padding=1)
        )

        # Projection head for contrastive learning
        self.projection_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(768, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

    def random_mask(self, x):
        B, C, H, W, D = x.shape
        mask = torch.rand(B, 1, H, W, D, device=x.device) > self.mask_ratio
        return x * mask, mask

    def forward(self, x):
        masked_x, mask = self.random_mask(x)
        hidden_states = self.encoder.swinViT(masked_x)
        latent = hidden_states[-1]  # deepest feature (B, 768, h, w, d)
        recon = self.decoder(latent)
        proj = self.projection_head(latent)
        return recon, proj, mask

In [ ]:
def mae_loss(recon, target, mask):
    diff = (recon - target) ** 2
    return (diff * (~mask)).sum() / (~mask).sum()

def contrastive_loss(z1, z2, temperature=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2*N, device=z.device, dtype=torch.bool)
    sim.masked_fill_(mask, -9e15)
    labels = torch.arange(N, device=z.device)
    labels = torch.cat([labels+N, labels], dim=0)
    return F.cross_entropy(sim, labels)

In [ ]:
def pretrain_step(model, imgs1, imgs2, optimizer, lambda_cont=0.1):
    recon1, proj1, mask1 = model(imgs1)
    recon2, proj2, mask2 = model(imgs2)

    loss_mae = mae_loss(recon1, imgs1, mask1) + mae_loss(recon2, imgs2, mask2)
    loss_cont = contrastive_loss(proj1, proj2)
    loss = loss_mae + lambda_cont * loss_cont

    optimizer.zero_grad()
    loss.backward()



    # 3. (Optional) Calculate Gradient Norm for Monitoring
    grad_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            # Calculate L2 norm for this parameter's gradient
            grad_norm += (p.grad.norm(2).item() ** 2)
    grad_norm = grad_norm ** 0.5

    optimizer.step()

    return loss.item(), loss_mae.item(), loss_cont.item(), grad_norm

In [ ]:
imgs = torch.randn(2, 1, 96, 96, 96).cuda()
recon, proj = model(imgs)
print(recon.shape, proj.shape)
print(proj.std().item(), proj.mean().item())

Load model and Loss

In [ ]:
import torch.nn.functional as F


def contrastive_loss(z1, z2, temperature=0.5):
    """
    SimCLR NT-Xent loss with proper masking and index handling.
    z1, z2: tensors of shape [N, D].
    """
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # [2N, D]
    z = F.normalize(z, dim=1)

    # Cosine similarity matrix
    sim = torch.matmul(z, z.T) / temperature
    sim_max, _ = sim.max(dim=1, keepdim=True)
    sim = sim - sim_max.detach()  # numerical stability

    # Mask self-similarity
    mask = torch.eye(2 * N, device=z.device, dtype=torch.bool)
    sim.masked_fill_(mask, -9e15)

    # Positive indices
    pos = torch.arange(N, device=z.device)
    pos_idx = torch.cat([pos + N, pos], dim=0)  # [2N]

    # Cross-entropy targets
    labels = pos_idx
    loss = F.cross_entropy(sim, labels)

    return loss

def pretrain_step(model, imgs1, imgs2, optimizer):
    recon1, proj1 = model(imgs1)
    recon2, proj2 = model(imgs2)

    # 1. Loss Calculations
    mae_loss = F.mse_loss(recon1, imgs1) + F.mse_loss(recon2, imgs2)
    cont_loss = contrastive_loss(proj1, proj2)
    loss = mae_loss + 0.1 * cont_loss

    # 2. Optimization
    optimizer.zero_grad()
    loss.backward()

    # 3. (Optional) Calculate Gradient Norm for Monitoring
    grad_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            # Calculate L2 norm for this parameter's gradient
            grad_norm += (p.grad.norm(2).item() ** 2)
    grad_norm = grad_norm ** 0.5

    optimizer.step()

    # Return individual losses and total loss item, plus the grad norm
    return loss.item(), mae_loss.item(), cont_loss.item(), grad_norm

Trainer Test

In [ ]:
dummy = torch.randn(1, 1, 96, 96, 96).to(device)
out = encoder(dummy)
print(out.shape)

In [ ]:
import os

model = DualPretrainModel().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
output_dir = "/content/drive/MyDrive/Διπλωματικη/runs"
os.makedirs(output_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {output_dir}")
for epoch in range(4):
    model.train()
    running_loss = 0.0
    running_mae_loss = 0.0
    running_cont_loss = 0.0
    running_grad_norm = 0.0

    # Track steps for printing frequency
    log_frequency = 10 # Print every 100 batches, adjust based on dataset size

    for batch_idx, batch in enumerate(train_loader):
        imgs = batch["image"].cuda()

        # Augmentations for contrastive pairs
        imgs1 = imgs + 0.05 * torch.randn_like(imgs)
        imgs2 = torch.flip(imgs, dims=[4])  # simple spatial flip

        # Call the updated pretrain_step
        total_loss, mae_loss, cont_loss, grad_norm = pretrain_step(model, imgs1, imgs2, optimizer)

        running_loss += total_loss
        running_mae_loss += mae_loss
        running_cont_loss += cont_loss
        running_grad_norm += grad_norm

        # Print progress periodically
        if (batch_idx + 1) % log_frequency == 0:
            avg_loss = running_loss / log_frequency
            avg_mae = running_mae_loss / log_frequency
            avg_cont = running_cont_loss / log_frequency
            avg_grad = running_grad_norm / log_frequency

            print(
                f"Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)} | "
                f"Total Loss: {avg_loss:.4f} | "
                f"MAE Loss: {avg_mae:.4f} | "
                f"Cont Loss: {avg_cont:.4f} | "
                f"Grad Norm: {avg_grad:.2f}"
            )

            # Reset running totals for the next log interval
            running_loss = 0.0
            running_mae_loss = 0.0
            running_cont_loss = 0.0
            running_grad_norm = 0.0

    # Print end-of-epoch summary for any remaining steps
    # (If the loop didn't end exactly on a log_frequency boundary)
    if running_loss > 0:
        remaining_batches = len(train_loader) % log_frequency
        if remaining_batches == 0:
            remaining_batches = log_frequency # Handle case where it was reset to 0

        avg_loss = running_loss / remaining_batches

        print(f"Epoch {epoch+1}, Remaining Avg Loss: {avg_loss:.4f}")

    print(f"--- Epoch {epoch+1} Complete ---")


    # 1. Define the filename: include the epoch number for tracking
    checkpoint_name = f"pretrained_checkpoint_epoch_{epoch+1:03d}.pth"
    save_path = os.path.join(output_dir, checkpoint_name)

    # 2. Create the checkpoint dictionary
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }

    # 3. Save the checkpoint to Google Drive
    torch.save(checkpoint, save_path)
    print(f"[SAVED] Checkpoint successfully saved to Google Drive: {save_path}")

final_save_path = os.path.join(output_dir, "final_pretrained_model_weights.pt")
torch.save(model.state_dict(), final_save_path) # Save just the weights for easy fine-tuning

print(f"\n--- Training Finished. Final model weights saved to: {final_save_path}")

TESTS

In [ ]:
imgs = next(iter(train_loader))["image"].cuda()
imgs1 = imgs + 0.05 * torch.randn_like(imgs)
imgs2 = torch.flip(imgs, dims=[4])
with torch.no_grad():
    recon, proj = model(imgs1)
print("recon:", recon.shape, "proj:", proj.shape)
print("proj mean/std:", proj.mean().item(), proj.std().item())


In [ ]:
total_loss, mae_loss, cont_loss, grad_norm = pretrain_step(model, imgs1, imgs2, optimizer)
print(mae_loss, cont_loss)

In [ ]:
!pip install -q "monai[all]" torch torchvision nibabel tqdm matplotlib

In [ ]:
import os
from glob import glob
import torch.nn.functional as F
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandFlipd, RandAffined, RandGaussianNoised,
    RandAdjustContrastd, RandSpatialCropd, CopyItemsd, EnsureTyped
)
from monai.data import Dataset, DataLoader, partition_dataset
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
# ------------------------------
# 1️⃣ Dataset setup
# ------------------------------
data_dir = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/volumes"
images = sorted(glob(os.path.join(data_dir, "*.nii.gz")))
data_dicts = [{"image": img} for img in images]
print(f"Found {len(data_dicts)} T2 scans.")

train_files, val_files = partition_dataset(
    data_dicts,
    ratios=[0.9, 0.1], # 90% training, 10% validation
    shuffle=True,      # Shuffle before splitting
    seed=42            # Use a fixed seed for reproducibility
)


print(f"Total scans: {len(data_dicts)}")
print(f"Training scans: {len(train_files)}")
print(f"Validation scans: {len(val_files)}")
# ------------------------------
# 2️⃣ MONAI SimCLR-style transforms
# ------------------------------
simclr_transform = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),

    ScaleIntensityRanged(
        keys=["image"], a_min=0, a_max=3000,
        b_min=0.0, b_max=1.0, clip=True
    ),

    RandSpatialCropd(keys=["image"], roi_size=(96, 96, 96), random_size=False),

    # 1. Create three copies: one for RoT, two for Contrastive
    CopyItemsd(keys=["image"], times=3, names=["image_rot", "image_con1", "image_con2"]),

    # --- Augmentations for ALL three views (Basic Non-Rotation) ---
    RandFlipd(keys=["image_rot", "image_con1", "image_con2"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandGaussianNoised(keys=["image_rot", "image_con1", "image_con2"], prob=0.3, mean=0.0, std=0.02),
    RandAdjustContrastd(keys=["image_rot", "image_con1", "image_con2"], prob=0.3, gamma=(0.7, 1.3)),


    # --- Selective Augmentations (Rotation for Contrastive ONLY) ---
    # NOTE: image_rot must NOT have random rotation, as the fixed 90-degree
    # rotation will be applied later in the training loop.
    RandAffined(
        keys=["image_con1"], prob=0.5,
        rotate_range=(0.1, 0.1, 0.1), # Random rotation applied
        scale_range=(0.1, 0.1, 0.1),
        mode="bilinear"
    ),
    RandAffined(
        keys=["image_con2"], prob=0.5,
        rotate_range=(0.1, 0.1, 0.1), # Random rotation applied
        scale_range=(0.1, 0.1, 0.1),
        mode="bilinear"
    ),

    EnsureTyped(keys=["image_rot", "image_con1", "image_con2"]),
])

# ------------------------------
# 3️⃣ Create dataset + dataloader
# ------------------------------
train_ds = Dataset(data=train_files, transform=simclr_transform)
val_ds = Dataset(data=val_files, transform=simclr_transform) # Use the same transform for validation

# Create DataLoaders
BATCH_SIZE = 2 # Assuming batch size of 1 for simplicity here
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2) # No need to shuffle validation


# ------------------------------
# 4️⃣ Visual confirmation
# ------------------------------
batch = next(iter(train_loader))
print(batch["image1"].shape, batch["image2"].shape)

# Pick a random slice (middle of the volume)
slice_idx = batch["image1"].shape[-1] // 2
img1 = batch["image1"][0, 0, :, :, slice_idx].cpu().numpy()
img2 = batch["image2"][0, 0, :, :, slice_idx].cpu().numpy()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap="gray")
plt.title("Augmented View 1 (image1)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img2, cmap="gray")
plt.title("Augmented View 2 (image2)")
plt.axis("off")

plt.suptitle("SimCLR-style augmentations for a single T2 MRI")
plt.show()





In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR


class MAEContrastivePretrainer(nn.Module):
    def __init__(self, projection_dim=128, mask_ratio=0.7):
        super().__init__()
        # use MONAI SwinUNETR backbone; out_channels=1 for reconstruction
        self.backbone = SwinUNETR(
            in_channels=1,
            out_channels=1,
            feature_size=96,
            use_checkpoint=True
        )
        self.projection_dim = projection_dim
        self.mask_ratio = mask_ratio

        # We will create channel_mapper and projector after we see the deepest feature shape
        self.channel_mapper = None
        self.projector = None

    def _init_heads(self, deepest_channels):
        # Called on first forward to make heads with correct dims
        D = deepest_channels
        mapped_dim = D # choose a reasonable mapped dim (you can change)
         #1. Channel Mapper (Optional 1x1 Conv)
        self.channel_mapper = nn.Conv3d(D, mapped_dim, kernel_size=1).to(next(self.parameters()).device)

        # 2. Projector (MLP only - Pooling/Flattening done explicitly in forward)
        self.projector = nn.Sequential(
            nn.Linear(mapped_dim, mapped_dim // 2),
            nn.ReLU(),
            nn.Linear(mapped_dim // 2, self.projection_dim)
        ).to(next(self.parameters()).device)
        self.mapped_dim = mapped_dim

    def random_mask(self, x):
        # x: [B,1,H,W,D]; mask by voxel probability
        mask = (torch.rand_like(x) > self.mask_ratio).float()  # 1=keep, 0=masked
        return x * mask, mask

    def forward(self, x):
        # 1) Reconstruction
        recon = self.backbone(x)

        # 2) Encoder Deepest Features
        hidden_states = self.backbone.swinViT(x)
        deepest = hidden_states[-1]  # shape [B, C_deep, d, h, w]

        if self.channel_mapper is None:
            self._init_heads(deepest.shape[1])

        mapped = self.channel_mapper(deepest)

        # 3) Global Pooling and Flattening (REQUIRED for Linear Layer)
        pooled = F.adaptive_avg_pool3d(mapped, (1, 1, 1)).view(x.shape[0], -1)

        # 4) Projection
        z = self.projector(pooled)
        z = F.normalize(z, dim=1)

        return recon, z, deepest # returning deepest for optional analysis/visualization

In [ ]:
# Cell: losses
def mae_mask_loss(recon, target, mask):
    """
    Recon & target: [B,1,H,W,D]; mask: 1=keep, 0=masked
    We compute L2 loss on masked voxels (i.e., where mask == 0), like MAE focusing on missing parts.
    """
    with torch.no_grad():
        masked_positions = (mask == 0).float()
        denom = masked_positions.sum()
        if denom.item() == 0:
            # degenerate case: no masked voxels; fallback to full recon loss
            return F.mse_loss(recon, target)
    diff = (recon - target)**2
    loss = (diff * masked_positions).sum() / denom
    return loss

def simclr_nt_xent(z, temperature=0.2):
    """
    Stable NT-Xent. z: [2N, D] (The already concatenated tensor)
    """
    N = z.size(0) // 2
    # z is already concatenated and normalized (or normalize again)
    z = F.normalize(z, dim=1)

    sim = torch.matmul(z, z.T) / temperature

    # remove self-similarity (large negative)
    diag_mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(diag_mask, -9e15)

    # positive indices (This logic is correct for the combined tensor)
    positives = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)], dim=0).to(z.device)

    loss = F.cross_entropy(sim, positives)
    return loss


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
# Cell: training step and loop
import torch
from tqdm import tqdm
import os
from monai.losses import ContrastiveLoss
from torch.cuda.amp import autocast
from torch.cuda.amp import GradScaler
# 1. Hyperparameters
GLOBAL_LAMBDA_CONTRAST = 0.01



contrast_loss_fn = ContrastiveLoss(temperature=0.1).to("cuda")

def pretrain_step(model, imgs1, imgs2, lambda_contrast=0.1, device="cuda"):
    """Computes losses and returns the loss tensor for backpropagation."""
    model.train()
    imgs1 = imgs1.to(device)
    imgs2 = imgs2.to(device)


    with autocast():
      # 1. Masking
      masked1, mask1 = model.random_mask(imgs1)
      masked2, mask2 = model.random_mask(imgs2)

      # 2. Forward passes
      # Note: Autograd is implicitly active here for backward() later
      recon1, z1, _ = model(masked1)
      recon2, z2, _ = model(masked2)
      z_combined = torch.cat([z1, z2], dim=0)

      batch_size = imgs1.shape[0] # The physical batch size B
      target_size = 2 * batch_size # 2*B

      # The target tensor (a tensor of zeros for this setup)
      target = torch.zeros(target_size, dtype=torch.long).to(z_combined.device)
      # 3. Losses

      loss_mae = mae_mask_loss(recon1, imgs1, mask1) + mae_mask_loss(recon2, imgs2, mask2)

      # FIX: Concatenate Z for correct NT-Xent calculation

      loss_contrast = contrast_loss_fn(z1,z2)

      # Total loss tensor (ready for backward())
      loss = loss_mae + GLOBAL_LAMBDA_CONTRAST * loss_contrast

    # Return loss tensor and itemized losses for logging
    return loss, loss_mae.item(), loss_contrast.item()

import os
from tqdm import tqdm
import gc # For cleaning up GPU memory

# --- Configuration ---
ACCUMULATION_STEPS = 32 # Simulates B_eff = 1 * 32 = 32. Adjust this to fit your Colab memory.
# ---------------------

GLOBAL_LAMBDA_CONTRAST = 0.1
def pretrain_loop(model, train_loader, epochs=20, lr=1e-4, save_dir="./runs", device="cuda"):

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scaler = GradScaler()
    os.makedirs(save_dir, exist_ok=True)
    model.to(device)


    optimizer.zero_grad(set_to_none=True) # Initialize gradients
    first_print = True

    for epoch in range(epochs):
        tot_l, tot_mae, tot_con = 0.0, 0.0, 0.0
        pbar = tqdm(train_loader, desc=f"Pretrain Epoch {epoch+1}/{epochs}")

        for i, batch in enumerate(pbar):

            # 1. Forward Pass and Loss Calculation
            # No need for .to(device) here, it's done inside the step function
            loss_tensor, l_mae, l_con = pretrain_step(
                model, batch["image1"], batch["image2"], device=device
            )

            # 2. Scale the loss for accumulation and backward pass
            loss_scaled = loss_tensor / ACCUMULATION_STEPS
            scaler.scale(loss_scaled).backward()

            # 3. Accumulate losses for logging (use unscaled loss tensor item)
            tot_l += loss_tensor.item()
            tot_mae += l_mae
            tot_con += l_con

            # 4. Optimization Step
            if (i + 1) % ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            # 5. Progress Bar Update (with unscaled running averages)
            pbar.set_postfix(
                total=f"{(tot_l / (i+1)):.4f}",
                mae=f"{(tot_mae / (i+1)):.4f}",
                cont=f"{(tot_con / (i+1)):.4f}"
            )

            # Debug shapes only on first iteration
            if first_print:
                # Add a print here to confirm the shape of z_combined for the first iteration
                print(f"B_eff={ACCUMULATION_STEPS}. SimCLR loss should now be > 0.")
                first_print = False

            torch.cuda.empty_cache()
            gc.collect()

        # Final step for any remaining accumulated gradients
        if (i + 1) % ACCUMULATION_STEPS != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # 6. End of Epoch Logging and Checkpoint Saving (Rest remains the same)
        avg_loss = tot_l / len(train_loader)
        avg_mae = tot_mae / len(train_loader)
        avg_con = tot_con / len(train_loader)
        print(f"[Epoch {epoch+1}] avg total: {avg_loss:.5f}  mae: {avg_mae:.5f}  cont: {avg_con:.5f}")

        ckpt = {
            "epoch": epoch+1,
            "model_state": model.state_dict(),
            "encoder_state": model.backbone.swinViT.state_dict()
        }
        save_path = os.path.join(save_dir, f"pretrain_epoch_{epoch+1:03d}.pt")
        torch.save(ckpt, save_path)
        print("Saved checkpoint:", save_path)


In [ ]:
for batch in train_loader:
    img = batch["image"]
    print("Sample batch image shape:", img.shape)  # expect (B, 1, H, W, D)
    break

In [ ]:
# Cell: run
EPOCHS = 10
LR = 1e-4
LAMBDA_CONTRAST = 0.1
SAVE_DIR = "/content/drive/MyDrive/Διπλωματικη/runs"  # adapt to your Drive
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
model = MAEContrastivePretrainer(projection_dim=128, mask_ratio=0.7)
pretrain_loop(model, train_loader, epochs=2, lr=1e-4)


In [ ]:
imgs = next(iter(train_loader))["image"].cuda()
with torch.no_grad():
    outputs = model.backbone(imgs)
    if isinstance(outputs, (list, tuple)):
        print("Encoder returned multiple outputs:")
        for i, o in enumerate(outputs):
            print(f"  Layer {i}: {tuple(o.shape)}")
    else:
        print("Encoder output shape:", tuple(outputs.shape))

In [ ]:
del(model)

In [ ]:
torch.cuda.empty_cache()

Hybrid SSL

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def simclr_nt_xent(z, temperature=0.5):


    """
    Corrected NT-Xent loss for paired contrastive SSL (Batch size N >= 2).
    Assumes z is L2-normalized and views are concatenated: [z1_view1, z1_view2, z2_view1, z2_view2, ...]
    """

    # z = [2N, C] where N is the original batch size (2 in your case)
    batch_size_2n = z.shape[0]
    batch_size_n = batch_size_2n // 2 # N = 2

    # 1. Calculate 2N x 2N similarity matrix (dot product of normalized vectors)
    # [4, 4] for your batch size of 2
    sim_matrix = torch.matmul(z, z.T) / temperature

    # 2. Identify Positive Pairs
    # The positives are the off-diagonal elements [i, i+N] and [i+N, i]
    # Example for N=2: (0, 2) and (2, 0) are positive, (1, 3) and (3, 1) are positive

    # Create the mask for positive pairs (Positives are at [i, i+N] and its transpose)
    pos_mask = torch.zeros(batch_size_2n, batch_size_2n, dtype=torch.bool, device=z.device)
    for i in range(batch_size_n):
        pos_mask[i, i + batch_size_n] = True
        pos_mask[i + batch_size_n, i] = True

    # Get the similarities of the positive pairs (Numerator term)
    pos_sim = sim_matrix[pos_mask].view(batch_size_2n, -1).squeeze(1)

    # 3. Mask out Self-Similarity (Diagonal) from the denominator
    # The total denominator (log-sum-exp) must exclude z_i against z_i
    diag_mask = torch.eye(batch_size_2n, dtype=torch.bool, device=z.device)
    SAFE_NEGATIVE = -10000.0
    sim_matrix = sim_matrix.masked_fill(diag_mask, SAFE_NEGATIVE)  # Set diagonal to a large negative number

    # 4. Compute Log-Sum-Exp (Denominator)
    # The denominator is log(sum(exp(sim_matrix / tau)))
    log_denom = torch.logsumexp(sim_matrix, dim=1)

    # 5. Compute Loss
    # Loss = -mean(pos_sim - log_denom)
    loss = -torch.mean(pos_sim - log_denom)

    return loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.losses import ContrastiveLoss
contrastive_loss_fn = ContrastiveLoss()


def ssl_loss_fn(outputs, imgs, lambda_mae=1.0, lambda_rot=0.5, lambda_con=0.1, lambda_light=0.5):
    loss = 0.0
    logs = {}

    # MAE reconstruction
    if "mae_recon" in outputs:
        l_mae = F.l1_loss(outputs["mae_recon"] * outputs["mask"], imgs * outputs["mask"])
        loss += lambda_mae * l_mae
        logs["mae"] = l_mae.item()

    # Rotation classification
    if "rot_logits" in outputs:
        l_rot = F.cross_entropy(outputs["rot_logits"], outputs["rot_labels"])
        loss += lambda_rot * l_rot
        logs["rot"] = l_rot.item()

    # Contrastive branch (SimCLR NT -Xent)
    if "proj" in outputs:
        z = outputs["proj"]
    # Split the concatenated features [2N, C] into two N-sized view batches
        N = z.shape[0] // 2
        z_view1 = z[:N]
        z_view2 = z[N:]

        # MONAI's ContrastiveLoss expects the two views as separate inputs
        # l_con = contrastive_loss_fn(z_view1, z_view2)

        # Note: If the tensor is [z1_v1, z1_v2, z2_v1, z2_v2], the split is more complex:
        # Assuming your forward returns [z1_v1, z2_v1, z1_v2, z2_v2]
        # Check the exact ordering of z. If it's [v1_b1, v1_b2, v2_b1, v2_b2, ...] then:
        # z_view1 = z[::2]
        # z_view2 = z[1::2]

        # Test the ordering and then use:
        l_con = contrastive_loss_fn(z_view1, z_view2)

        loss += lambda_con * l_con
        logs["contrastive"] = l_con.item()

    # Light decoder reconstruction
    if "light_recon" in outputs:
        l_light = F.smooth_l1_loss(outputs["light_recon"], imgs)
        loss += lambda_light * l_light
        logs["light_recon"] = l_light.item()

    return loss, logs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
from typing import Sequence

class HybridSSLPretrainer(nn.Module):
    def __init__(self,
                 in_channels=1,
                 feature_size=48, # <-- C = 48
                 projection_dim=128,
                 mask_ratio=0.7,
                 use_rotation=True,
                 use_contrastive=True,
                 use_reconstruction=True):
        super().__init__()

        # --- RE-CALCULATE CORRECT DIMENSIONS ---
        # 1. Deepest/Pooled Channel Count (16 * C): 16 * 48 = 768
        ACTUAL_POOLED_CHANNELS = 16 * feature_size # 768

        # 2. Intermediate Projection Channel Count (4 * C): 4 * 48 = 192
        INTERMEDIATE_CHANNELS = 4 * feature_size # 192

        # === Swin backbone (UNETR encoder+decoder for later fine-tuning) ===
        self.backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=in_channels,
            feature_size=feature_size,
            use_checkpoint=True
        )

        # === SSL options (remains unchanged) ===
        self.mask_ratio = mask_ratio
        self.use_rotation = use_rotation
        self.use_contrastive = use_contrastive
        self.use_reconstruction = use_reconstruction

        # === Projection head (for contrastive branch) ===
        self.projector = nn.Sequential(
            # FIX 1: Input size must be 768 (ACTUAL_POOLED_CHANNELS)
            nn.Linear(ACTUAL_POOLED_CHANNELS, INTERMEDIATE_CHANNELS), # 768 -> 192
            nn.ReLU(),
            nn.Linear(INTERMEDIATE_CHANNELS, projection_dim)          # 192 -> 128
        )

        # === Rotation head (predict rotation out of 4 classes) ===
        # FIX 2: Input size must be 768 (ACTUAL_POOLED_CHANNELS)
        self.rot_head = nn.Linear(ACTUAL_POOLED_CHANNELS, 4) # 768 -> 4

        # === Lightweight VAE-like reconstruction decoder ===
        # Assuming feature_size = 48 (C)
          # 16C
        C4 = 192
        C2 = 96
        C1 = 48 # New intermediate channel size

        self.light_decoder = nn.Sequential(
            # 1. Goes from 3x3x3 to 6x6x6
            nn.Conv3d(ACTUAL_POOLED_CHANNELS, C4, 3, padding=1),  # 768 -> 192
            nn.InstanceNorm3d(C4), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            # 2. Goes from 6x6x6 to 12x12x12
            nn.Conv3d(C4, C2, 3, padding=1),  # 192 -> 96
            nn.InstanceNorm3d(C2), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            # --- NEW: Step 3. Goes from 12x12x12 to 24x24x24 ---
            nn.Conv3d(C2, C1, 3, padding=1),  # 96 -> 48
            nn.InstanceNorm3d(C1), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            # --- NEW: Step 4. Goes from 24x24x24 to 48x48x48 ---
            nn.Conv3d(C1, C1, 3, padding=1),  # 48 -> 48
            nn.InstanceNorm3d(C1), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            # --- NEW: Step 5. Goes from 48x48x48 to 96x96x96 ---
            nn.Conv3d(C1, C1, 3, padding=1),  # 48 -> 48
            nn.InstanceNorm3d(C1), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            # FINAL: Output layer (96x96x96)
            nn.Conv3d(C1, in_channels, 1)
        )

    # --- utility: random voxel masking (remains unchanged) ---
    def random_mask(self, x):
        mask = (torch.rand_like(x) > self.mask_ratio).float()
        return x * mask, mask

    # --- utility: 3D rotation augmentation (remains unchanged) ---
    def rotate_3d(self, x, k):
        return torch.rot90(x, k, dims=(-2, -1))

    # --- forward method (remains unchanged) ---
    def forward(self, x, rotation_labels=None):
        out = {}
        x = x.contiguous()

        # (1) Random masking for MAE-style reconstruction
        masked_x, mask = self.random_mask(x)

        # (2) Forward backbone (reconstruction UNETR)
        recon = self.backbone(masked_x)
        out["mae_recon"] = recon
        out["mask"] = mask

        # (3) Deepest features for global SSL heads
        hidden_states = self.backbone.swinViT(x)
        deepest = hidden_states[-1]  # [B, C=768, D, H, W]
        pooled = F.adaptive_avg_pool3d(deepest, 1).flatten(1)  # [B, C=768]

        # (4) Rotation branch
        if self.use_rotation and rotation_labels is not None:
            out["rot_logits"] = self.rot_head(pooled)
            out["rot_labels"] = rotation_labels

        # (5) Contrastive projection branch
        if self.use_contrastive:
            z = F.normalize(self.projector(pooled), dim=1)
            out["proj"] = z

        # (6) Lightweight reconstruction branch
        if self.use_reconstruction:
            out["light_recon"] = self.light_decoder(deepest)

        return out

In [ ]:

import torch
from tqdm import tqdm
import os
from monai.losses import ContrastiveLoss
from torch.cuda.amp import autocast
from torch.cuda.amp import GradScaler


def pretrain_step(model, batch, optimizer, scaler, device):
    model.train()
    imgs = batch["image"].to(device)

    # --- Random rotation augmentation ---
    k = torch.randint(0, 4, (imgs.size(0),), device=device)
    rotated_imgs = torch.stack([model.rotate_3d(imgs[i], int(k[i])) for i in range(imgs.size(0))])

    # --- Forward + backward under AMP ---
    with autocast():
        outputs = model(rotated_imgs, rotation_labels=k)
        loss, logs = ssl_loss_fn(outputs, imgs)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    logs["total"] = loss.item()
    return logs

In [ ]:
def evaluate_ssl(model, val_loader, device):
    model.eval()
    total_loss = mae_loss = rot_loss = con_loss = light_loss = 0.0
    rot_correct = rot_total = count = 0

    with autocast():
        for batch in val_loader:
            imgs = batch["image"].to(device)
            k = torch.randint(0, 4, (imgs.size(0),), device=device)
            rotated_imgs = torch.stack([model.rotate_3d(imgs[i], int(k[i])) for i in range(imgs.size(0))])

            outputs = model(rotated_imgs, rotation_labels=k)
            loss, logs = ssl_loss_fn(outputs, imgs)

            total_loss += loss.item()
            mae_loss += logs.get("mae", 0)
            rot_loss += logs.get("rot", 0)
            con_loss += logs.get("contrastive", 0)
            light_loss += logs.get("light_recon", 0)
            count += 1

            if "rot_logits" in outputs:
                preds = outputs["rot_logits"].argmax(dim=1)
                rot_correct += (preds == outputs["rot_labels"]).sum().item()
                rot_total += preds.numel()

    metrics = {
        "total_loss": total_loss / count,
        "mae_loss": mae_loss / count,
        "rot_loss": rot_loss / count,
        "contrastive_loss": con_loss / count,
        "light_recon_loss": light_loss / count,
        "rotation_acc": rot_correct / rot_total if rot_total > 0 else None,
    }
    return metrics

In [ ]:
from tqdm import tqdm
import torch
import os
import gc
from torch.utils.tensorboard import SummaryWriter

def pretrain_loop(model, train_loader, val_loader=None,
                  epochs=20, lr=1e-4, device="cuda",
                  save_dir="/content/drive/MyDrive/Διπλωματικη/runs/ssl", log_dir="/content/drive/MyDrive/Διπλωματικη/runs/ssl_logs"):

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scaler = GradScaler()
    model.to(device)

    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        print(f"\n===== Epoch {epoch}/{epochs} =====")
        model.train()
        epoch_logs = {"mae": 0, "rot": 0, "contrastive": 0, "light_recon": 0, "total": 0}
        count = 0

        # === Training ===
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch}"):
            logs = pretrain_step(model, batch, optimizer, scaler, device)
            for k, v in logs.items():
                epoch_logs[k] = epoch_logs.get(k, 0) + v
            count += 1

            # memory hygiene
            torch.cuda.empty_cache()
            gc.collect()

        # Average logs
        for k in epoch_logs:
            epoch_logs[k] /= count
            writer.add_scalar(f"train/{k}", epoch_logs[k], epoch)

        print(f"Train | " + " | ".join([f"{k}: {v:.4f}" for k, v in epoch_logs.items()]))

        # === Validation ===
        if val_loader is not None:
            val_metrics = evaluate_ssl(model, val_loader, device)
            for k, v in val_metrics.items():
                if v is not None:
                    writer.add_scalar(f"val/{k}", v, epoch)
            print(f"Val   | " + " | ".join([f"{k}: {v:.4f}" for k, v in val_metrics.items() if v is not None]))

            val_loss = val_metrics["total_loss"]
        else:
            val_loss = epoch_logs["total"]

        # === Checkpoint ===
        ckpt = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": val_loss,
        }
        save_path = os.path.join(save_dir, f"epoch_{epoch:03d}.pt")
        torch.save(ckpt, save_path)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(save_dir, "best_model.pt")
            torch.save(ckpt, best_path)
            print(f"✅ New best checkpoint saved at {best_path}")
        else:
            print(f"Saved checkpoint at {save_path}")

    writer.close()
    print("🏁 Training complete. Best validation loss:", best_val_loss)

In [ ]:
# Cell: run
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridSSLPretrainer()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

pretrain_loop(
    model=model,
    train_loader=train_loader,
    device=device,
    epochs=10,
    val_loader=val_loader

)

In [ ]:
!pip install -q "monai[all]" torch torchvision nibabel tqdm matplotlib

In [ ]:
import os
from glob import glob
import torch.nn.functional as F
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandFlipd, RandAffined, RandGaussianNoised,
    RandAdjustContrastd, RandSpatialCropd, CopyItemsd, EnsureTyped
)
from monai.data import Dataset, DataLoader, partition_dataset
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
# ------------------------------
# 1️⃣ Dataset setup
# ------------------------------
data_dir = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/volumes"
images = sorted(glob(os.path.join(data_dir, "*.nii.gz")))
data_dicts = [{"image": img} for img in images]
print(f"Found {len(data_dicts)} T2 scans.")

train_files, val_files = partition_dataset(
    data_dicts,
    ratios=[0.8, 0.2], # 90% training, 10% validation
    shuffle=True,      # Shuffle before splitting
    seed=42            # Use a fixed seed for reproducibility
)


print(f"Total scans: {len(data_dicts)}")
print(f"Training scans: {len(train_files)}")
print(f"Validation scans: {len(val_files)}")
# ------------------------------
# 2️⃣ MONAI SimCLR-style transforms
# ------------------------------
simclr_transform = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),

    ScaleIntensityRanged(
        keys=["image"], a_min=0, a_max=3000,
        b_min=0.0, b_max=1.0, clip=True
    ),

    RandSpatialCropd(keys=["image"], roi_size=(96, 96, 96), random_size=False),

    # 1. Create three copies: one for RoT, two for Contrastive
    CopyItemsd(keys=["image"], times=3, names=["image_rot", "image_con1", "image_con2"]),

    # --- Augmentations for ALL three views (Basic Non-Rotation) ---
    RandFlipd(keys=["image_rot", "image_con1", "image_con2"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandGaussianNoised(keys=["image_rot", "image_con1", "image_con2"], prob=0.3, mean=0.0, std=0.02),
    RandAdjustContrastd(keys=["image_rot", "image_con1", "image_con2"], prob=0.3, gamma=(0.7, 1.3)),


    # --- Selective Augmentations (Rotation for Contrastive ONLY) ---
    # NOTE: image_rot must NOT have random rotation, as the fixed 90-degree
    # rotation will be applied later in the training loop.
    RandAffined(
        keys=["image_con1"], prob=0.5,
        rotate_range=(0.1, 0.1, 0.1),
        scale_range=(0.1, 0.1, 0.1),
        mode="bilinear"
    ),
    RandAffined(
        keys=["image_con2"], prob=0.5,
        rotate_range=(0.1, 0.1, 0.1),
        scale_range=(0.1, 0.1, 0.1),
        mode="bilinear"
    ),

    EnsureTyped(keys=["image_rot", "image_con1", "image_con2"]),
])

# ------------------------------
# 3️⃣ Create dataset + dataloader
# ------------------------------
train_ds = Dataset(data=train_files, transform=simclr_transform)
val_ds = Dataset(data=val_files, transform=simclr_transform) # Use the same transform for validation

# Create DataLoaders
# BATCH_SIZE = 2 is correct for the minimum N of the contrastive loss
BATCH_SIZE = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2) # No need to shuffle validation


# ------------------------------
# 4️⃣ Visual confirmation (Updated for the new keys)
# ------------------------------
batch = next(iter(train_loader))

# Check the shapes of the keys
print(f"Original View shape: {batch['image'].shape}")
print(f"Rotation View shape: {batch['image_rot'].shape}")
print(f"Contrastive View 1 shape: {batch['image_con1'].shape}")
print(f"Contrastive View 2 shape: {batch['image_con2'].shape}")


# Pick a random slice (using image_con1 as reference)
slice_idx = batch["image_con1"].shape[-1] // 2

# Extract the views for plotting
# Note: The original image is also spatially cropped to 96x96x96 by the transform
img_orig = batch["image"][0, 0, :, :, slice_idx].cpu().numpy()
img_rot = batch["image_rot"][0, 0, :, :, slice_idx].cpu().numpy()
img_con1 = batch["image_con1"][0, 0, :, :, slice_idx].cpu().numpy()
img_con2 = batch["image_con2"][0, 0, :, :, slice_idx].cpu().numpy()

# Use a 2x2 grid for visualization
plt.figure(figsize=(12, 10))

# --- Subplot 1: Original Cropped View ---
plt.subplot(2, 2, 1)
plt.imshow(img_orig, cmap="gray")
plt.title("1. Original Cropped (Base Image)")
plt.axis("off")

# --- Subplot 2: RoT View ---
# Should show non-affine augmentations (flip, noise, contrast) but NO random rotation.
plt.subplot(2, 2, 2)
plt.imshow(img_rot, cmap="gray")
plt.title("2. RoT View (Augmented, NO Affine Rotation)")
plt.axis("off")

# --- Subplot 3: Contrastive View 1 ---
# Should show affine augmentations (random rotation, scaling).
plt.subplot(2, 2, 3)
plt.imshow(img_con1, cmap="gray")
plt.title("3. Contrastive View 1 (Augmented + Affine)")
plt.axis("off")

# --- Subplot 4: Contrastive View 2 ---
# Should show a DIFFERENT set of affine augmentations.
plt.subplot(2, 2, 4)
plt.imshow(img_con2, cmap="gray")
plt.title("4. Contrastive View 2 (Augmented + Affine)")
plt.axis("off")

plt.suptitle("Four-View Setup Confirmation", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
from monai.losses import ContrastiveLoss
import gc
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm


def mae_mask_loss(pred, target, mask):
    """
    MAE loss (L1) focusing on the masked/reconstructed regions.
    The mask here is assumed to be the binary mask (1.0 or 0.0) where 1.0 means 'masked'.
    """

    masked_target = target * mask
    masked_pred = pred * mask

    # Normalization by the number of masked voxels
    num_masked_voxels = mask.sum()

    # Return L1 loss (MAE) summed over the masked regions
    return F.l1_loss(masked_pred, masked_target, reduction='sum') / (num_masked_voxels + 1e-8)


class HybridSSLPretrainer(nn.Module):
    def __init__(self,
                 in_channels=1,
                 feature_size=48,
                 projection_dim=128,
                 mask_ratio=0.7,
                 use_rotation=True,
                 use_contrastive=True,
                 use_reconstruction=True):
        super().__init__()

        # Derived dimensions
        ACTUAL_POOLED_CHANNELS = 16 * feature_size  # 768
        INTERMEDIATE_CHANNELS = 4 * feature_size   # 192

        # === Swin backbone (UNETR encoder+decoder) ===
        self.backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=in_channels,
            feature_size=feature_size,
            use_checkpoint=True
        )

        self.mask_ratio = mask_ratio
        self.use_rotation = use_rotation
        self.use_contrastive = use_contrastive
        self.use_reconstruction = use_reconstruction

        # === Projection head (Contrastive) ===
        self.projector = nn.Sequential(
            nn.Linear(ACTUAL_POOLED_CHANNELS, INTERMEDIATE_CHANNELS),
            nn.ReLU(),
            nn.Linear(INTERMEDIATE_CHANNELS, projection_dim)
        )

        # === Rotation head (4 classes) ===
        self.rot_head = nn.Linear(ACTUAL_POOLED_CHANNELS, 4)

        # === Lightweight VAE-like reconstruction decoder ===
        C = feature_size
        self.light_decoder = nn.Sequential(
            nn.Conv3d(ACTUAL_POOLED_CHANNELS, 8*C, 3, padding=1),
            nn.InstanceNorm3d(8*C), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            nn.Conv3d(8*C, 4*C, 3, padding=1),
            nn.InstanceNorm3d(4*C), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            nn.Conv3d(4*C, 2*C, 3, padding=1),
            nn.InstanceNorm3d(2*C), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            nn.Conv3d(2*C, C, 3, padding=1),
            nn.InstanceNorm3d(C), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            nn.Conv3d(C, C, 3, padding=1),
            nn.InstanceNorm3d(C), nn.LeakyReLU(),
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),

            nn.Conv3d(C, in_channels, 1)
        )

    # --- utility: random voxel masking ---
    def random_mask(self, x):
        # mask is 1.0 where unmasked, 0.0 where masked
        unmasked_mask = (torch.rand_like(x) > self.mask_ratio).float()
        # The true binary mask for MAE (1.0 where masked, 0.0 where unmasked)
        mae_mask = 1.0 - unmasked_mask
        return x * unmasked_mask, mae_mask

    # --- utility: 3D rotation augmentation ---
    def rotate_3d(self, x, k):
        return torch.rot90(x, k, dims=(-2, -1))

    # --- FORWARD METHOD (Handles all 4 SSL branches) ---
    def forward(self, x_mae: torch.Tensor, x_con1: torch.Tensor, x_con2: torch.Tensor, x_rot: torch.Tensor, rotation_labels: torch.Tensor):
        out = {}

        # 1. MAE Reconstruction (Requires MASKED input)
        out["mae_recon"] = self.backbone(x_mae.contiguous())

        # 2. Feature Extraction for Contrastive & Light Recon (Base: x_con1)
        hidden_states_con1 = self.backbone.swinViT(x_con1.contiguous())
        deepest_con1 = hidden_states_con1[-1]
        pooled_con1 = F.adaptive_avg_pool3d(deepest_con1, 1).flatten(1)

        # 3. Feature Extraction for Rotation Task (Input: x_rot)
        if self.use_rotation and rotation_labels is not None:
            rot_states = self.backbone.swinViT(x_rot.contiguous())
            pooled_rot = F.adaptive_avg_pool3d(rot_states[-1], 1).flatten(1)
            out["rot_logits"] = self.rot_head(pooled_rot)
            out["rot_labels"] = rotation_labels

        # 4. Feature Extraction for Contrastive View 2 (Input: x_con2)
        if self.use_contrastive:
            out["proj_con1"] = F.normalize(self.projector(pooled_con1), dim=1)
            con2_states = self.backbone.swinViT(x_con2.contiguous())
            pooled_con2 = F.adaptive_avg_pool3d(con2_states[-1], 1).flatten(1)
            out["proj_con2"] = F.normalize(self.projector(pooled_con2), dim=1)

        # 5. Lightweight reconstruction branch
        if self.use_reconstruction:
            out["light_recon"] = self.light_decoder(deepest_con1)

        return out

In [ ]:
from monai.losses import ContrastiveLoss

# --- GLOBAL CONFIGURATION ---
GLOBAL_LAMBDA_ROT = 0
GLOBAL_LAMBDA_CONTRAST = 1
GLOBAL_LAMBDA_LIGHT_RECON = 0.1

contrast_loss_fn = ContrastiveLoss(temperature=0.5).to("cuda")

# --- CORRECTED TRAINING STEP ---
def pretrain_step(model, batch, device="cuda"):
    """
    Performs one step of the pretraining forward pass and loss calculation.
    Returns the loss tensor (for backward) and a logs dictionary.
    """
    model.train()

    # 1. Prepare Inputs
    x_rot_clean = batch["image_rot"].to(device)
    x_con1 = batch["image_con1"].to(device)
    x_con2 = batch["image_con2"].to(device)


    # 2. Setup Rotation Task
    k = torch.randint(4, (x_rot_clean.shape[0],), dtype=torch.long).to(device) # [B]
    # Use a list comprehension to apply the unique rotation k[i] to each item x_rot_clean[i]
    x_rot = torch.stack([model.rotate_3d(x_rot_clean[i], int(k[i]))
                        for i in range(x_rot_clean.size(0))])

    # 3. Setup MAE Masking
    x_mae, mae_mask = model.random_mask(x_con1) # mae_mask is the binary mask (1 where masked)

    with autocast():
        # 4. Forward Pass
        outputs = model(
            x_mae=x_mae,
            x_con1=x_con1,
            x_con2=x_con2,
            x_rot=x_rot,
            rotation_labels=k
        )

        # 5. Loss Calculation
        l_mae = mae_mask_loss(outputs["mae_recon"], x_con1, mae_mask)
        l_rot = F.cross_entropy(outputs["rot_logits"], k)
        l_con = contrast_loss_fn(outputs["proj_con1"], outputs["proj_con2"])
        l_light = F.smooth_l1_loss(outputs["light_recon"], x_con1)

        # Total Loss Tensor
        loss_tensor = l_mae + (GLOBAL_LAMBDA_ROT * l_rot) + \
                      (GLOBAL_LAMBDA_CONTRAST * l_con) + \
                      (GLOBAL_LAMBDA_LIGHT_RECON * l_light)

    logs = {
        "mae": l_mae.item(),
        "rot": l_rot.item(),
        "contrastive": l_con.item(),
        "light_recon": l_light.item(),
        "total": loss_tensor.item()
    }

    return loss_tensor, logs


# --- CORRECTED EVALUATION FUNCTION ---
def evaluate_ssl(model, val_loader, device):
    model.eval()

    total_loss = mae_loss = rot_loss = con_loss = light_loss = 0.0
    rot_correct = rot_total = count = 0

    with torch.no_grad():
        # Use the updated autocast syntax
        with autocast():
            pbar = tqdm(val_loader, desc=f"Validation")
            for batch in pbar:

                # 1. LOSS CALCULATION
                # Re-use pretrain_step logic for forward pass and loss accumulation
                # (pretrain_step is assumed to contain the rotation loop fix for x_rot)
                loss_tensor, logs = pretrain_step(model, batch, device)

                # Metric Accumulation
                total_loss += logs["total"]
                mae_loss += logs["mae"]
                rot_loss += logs["rot"]
                con_loss += logs["contrastive"]
                light_loss += logs["light_recon"]
                count += 1

                # 2. ROTATION ACCURACY CALCULATION
                # NOTE: We have to re-run the rotation setup here because the rotation
                # index (k) and x_rot are generated RANDOMLY inside pretrain_step
                # and are not stored/returned in a reusable format for accuracy calculation.

                x_rot_clean = batch["image_rot"].to(device)
                k = torch.randint(4, (x_rot_clean.shape[0],), dtype=torch.long).to(device)

                # --- FIX: Apply rotation via list comprehension (as provided by user) ---
                x_rot = torch.stack([model.rotate_3d(x_rot_clean[i], int(k[i]))
                                     for i in range(x_rot_clean.size(0))])
                # -----------------------------------------------------------------------

                # Forward pass through encoder and RoT head for accuracy check
                rot_states = model.backbone.swinViT(x_rot.contiguous())
                pooled_rot = F.adaptive_avg_pool3d(rot_states[-1], 1).flatten(1)
                rot_logits = model.rot_head(pooled_rot)

                preds = rot_logits.argmax(dim=1)
                rot_correct += (preds == k).sum().item()
                rot_total += preds.numel()

                pbar.set_postfix(val_total=f"{total_loss / count:.4f}", rot_acc=f"{rot_correct / rot_total:.4f}")

    metrics = {
        "total_loss": total_loss / count,
        "mae_loss": mae_loss / count,
        "rot_loss": rot_loss / count,
        "contrastive_loss": con_loss / count,
        "light_recon_loss": light_loss / count,
        "rotation_acc": rot_correct / rot_total if rot_total > 0 else 0.0,
    }
    return metrics

In [ ]:
import torch
import torch.nn as nn
import os
from tqdm import tqdm
import gc
from torch.cuda.amp import GradScaler
from torch.utils.tensorboard import SummaryWriter # Assuming you might want this later

ACCUMULATION_STEPS = 32 # Keep this global or pass it in too

# --- ADD optimizer, scaler, start_epoch, initial_best_val_loss to the definition ---
def pretrain_loop(model, train_loader, val_loader, optimizer, scaler, start_epoch, initial_best_val_loss,
                  epochs=20, lr=1e-4, device="cuda",
                  save_dir="/content/drive/MyDrive/Διπλωματικη/runs/ssl", log_dir="/content/drive/MyDrive/Διπλωματικη/runs/ssl_logs"):

    # --- REMOVE internal initialization of these ---
    # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5) # REMOVED
    # scaler = GradScaler() # REMOVED
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True) # Ensure log dir is created
    # writer = SummaryWriter(log_dir) # Uncomment if using TensorBoard

    model.to(device) # Ensure model is on the correct device

    # --- USE the passed-in value ---
    best_val_loss = initial_best_val_loss

    # optimizer.zero_grad(set_to_none=True) # Initial zero_grad happens before the loop starts now

    first_print = True # Keep this for the B_eff message

    # --- Adjust the epoch range ---
    for epoch in range(start_epoch, epochs + 1):
        print(f"\n===== Epoch {epoch}/{epochs} =====")
        model.train()
        epoch_logs = {"mae": 0, "rot": 0, "contrastive": 0, "light_recon": 0, "total": 0}
        count = 0

        # === Training ===
        pbar = tqdm(train_loader, desc=f"Training Epoch {epoch}")
        for i, batch in enumerate(pbar):
            # 1. Forward Pass and Loss Calculation

            loss_tensor, logs = pretrain_step(model, batch, device=device)

            # 2. Scale loss and backward
            loss_scaled = loss_tensor / ACCUMULATION_STEPS
            scaler.scale(loss_scaled).backward()

            # 3. Accumulate logs
            for k, v in logs.items():
                epoch_logs[k] = epoch_logs.get(k, 0) + v
            count += 1 # Increment count here

            # 4. Optimization Step
            if (i + 1) % ACCUMULATION_STEPS == 0:
                # Clip gradients BEFORE scaler.step()
                scaler.unscale_(optimizer) # Unscale gradients first
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip
                scaler.step(optimizer) # Optimizer step uses unscaled gradients
                scaler.update() # Update scaler
                optimizer.zero_grad(set_to_none=True) # Reset gradients

            # 5. Progress Bar Update
            # Update using accumulated values over 'count' steps within the epoch
            pbar.set_postfix({k: f"{(v / count):.4f}" for k, v in epoch_logs.items()})


            if first_print:
                print(f"Effective Batch Size (B_eff) = {ACCUMULATION_STEPS * train_loader.batch_size}.")
                first_print = False

            torch.cuda.empty_cache()
            gc.collect()
        # --- End Batch Loop ---

        # Final step for remaining gradients
        if count % ACCUMULATION_STEPS != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # 6. End of Epoch Logging and Validation
        # Average logs over the total number of steps in the epoch (count)
        for k in epoch_logs:
            epoch_logs[k] /= count
            # writer.add_scalar(f"train/{k}", epoch_logs[k], epoch) # Uncomment for TensorBoard

        print(f"Train | " + " | ".join([f"{k}: {v:.4f}" for k, v in epoch_logs.items()]))


        # === Validation ===
        if val_loader is not None:
            val_metrics = evaluate_ssl(model, val_loader, device)
            # Write val metrics to TensorBoard...
            # for k, v in val_metrics.items():
            #    if v is not None: writer.add_scalar(f"val/{k}", v, epoch)
            print(f"Val   | " + " | ".join([f"{k}: {v:.4f}" for k, v in val_metrics.items() if v is not None]))
            val_loss = val_metrics["total_loss"]
        else:
            val_loss = epoch_logs["total"] # Use train loss if no validation

        # === Checkpoint ===
        ckpt = {
            'epoch': epoch, # Save the *completed* epoch number
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scaler_state': scaler.state_dict(),
            'val_loss': val_loss,
            'encoder_state': model.backbone.swinViT.state_dict() # Optional
        }

        latest_save_path = os.path.join(save_dir, f"pretrain_epoch_{epoch:03d}.pt")
        torch.save(ckpt, latest_save_path)
        print(f"Saved latest checkpoint: {latest_save_path}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(save_dir, "pretrain_BEST.pt")
            torch.save(ckpt, best_path)
            print(f"✅ New best checkpoint saved at {best_path} (Val Loss: {best_val_loss:.5f})")

    # writer.close() # Uncomment for TensorBoard
    print("🏁 Training complete. Best validation loss:", best_val_loss)

In [ ]:
import torch
import os
from torch.cuda.amp import GradScaler # Make sure GradScaler is imported here too

# --- CONFIGURATION ---
ACCUMULATION_STEPS = 32
best_checkpoint_path = "/content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt" # Path to load from
device = "cuda" if torch.cuda.is_available() else "cpu"
start_epoch = 1 # Default starting epoch
best_val_loss = float('inf') # Default best loss
TOTAL_EPOCHS = 20 # Total epochs you want to train for
LR = 1e-4

print("Device:", device)

# --- Initialize Model, Optimizer, Scaler ---
model = HybridSSLPretrainer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5) # Initialize optimizer first
scaler = GradScaler()
# --- Load Checkpoint (if exists) ---
if os.path.exists(best_checkpoint_path):
    print(f"Loading checkpoint from: {best_checkpoint_path}")
    checkpoint = torch.load(best_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state'])
    if 'optimizer_state' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state'])
    # Load scaler state if it was saved in the checkpoint you are loading
    if 'scaler_state' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state'])
    if 'epoch' in checkpoint:
        start_epoch = checkpoint['epoch'] + 1 # Start from the next epoch
    if 'val_loss' in checkpoint:
        best_val_loss = checkpoint['val_loss']

    print(f"Resuming training from epoch {start_epoch}, best previous val_loss: {best_val_loss:.5f}")
else:
    print("No checkpoint found, starting training from scratch.")



pretrain_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer, # Pass the potentially loaded optimizer
    scaler=scaler,       # Pass the potentially loaded scaler
    start_epoch=start_epoch, # Pass the determined start epoch
    initial_best_val_loss=best_val_loss, # Pass the loaded best loss
    epochs=TOTAL_EPOCHS,
    lr=LR, # Pass LR if pretrain_loop uses it internally (though optimizer already has it)
    device=device
)

In [ ]:
import gc
gc.collect()

In [ ]:
torch.cuda.empty_cache()

 `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.

Pretrain Epoch 1/10: 0%| | 0/198 [00:00<?, ?it/s]`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.

Pretrain Epoch 1/10: 0%| | 0/198 [00:05<?, ?it/s, con=1.0950, light=0.0720, mae=0.3608, rot=1.8638, total=1.8474]Effective Batch Size (B_eff) = 64.Pretrain Epoch 1/10: 100%|██████████| 198/198 [11:57<00:00, 3.62s/it, con=1.0888, light=0.0515, mae=0.2501, rot=2.7673, total=2.1832]

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.

Validation: 100%|██████████| 50/50 [01:07<00:00, 1.35s/it, rot_acc=0.1515, val_total=1.6963]--------------------------------------------------

[Epoch 1 TRAIN] total: 2.18325 | mae: 0.25006 | rot: 2.76726 | cont: 1.08881 | light: 0.05150

[Epoch 1 VALID] total: 1.69631 | rot_acc: 0.1515

--------------------------------------------------

Saved BEST checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_001.ptPretrain Epoch 2/10: 100%|██████████| 198/198 [11:59<00:00, 3.63s/it, con=1.0941, light=0.0137, mae=0.1568, rot=1.7285, total=1.5694]

Validation: 100%|██████████| 50/50 [01:07<00:00, 1.36s/it, rot_acc=0.2222, val_total=1.3675]--------------------------------------------------

[Epoch 2 TRAIN] total: 1.56944 | mae: 0.15678 | rot: 1.72847 | cont: 1.09410 | light: 0.01373

[Epoch 2 VALID] total: 1.36750 | rot_acc: 0.2222

--------------------------------------------------

Saved BEST checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_002.ptPretrain Epoch 3/10: 100%|██████████| 198/198 [12:01<00:00, 3.64s/it, con=1.0922, light=0.0107, mae=0.1229, rot=1.4598, total=1.4000]

Validation: 100%|██████████| 50/50 [01:06<00:00, 1.33s/it, rot_acc=0.3232, val_total=1.3393]--------------------------------------------------

[Epoch 3 TRAIN] total: 1.40002 | mae: 0.12290 | rot: 1.45985 | cont: 1.09224 | light: 0.01074

[Epoch 3 VALID] total: 1.33928 | rot_acc: 0.3232

--------------------------------------------------

Saved BEST checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_003.ptPretrain Epoch 4/10: 100%|██████████| 198/198 [12:01<00:00, 3.64s/it, con=1.0667, light=0.0097, mae=0.1013, rot=1.4120, total=1.3416]

Validation: 100%|██████████| 50/50 [01:06<00:00, 1.34s/it, rot_acc=0.2020, val_total=1.2625]--------------------------------------------------

[Epoch 4 TRAIN] total: 1.34164 | mae: 0.10133 | rot: 1.41198 | cont: 1.06671 | light: 0.00974

[Epoch 4 VALID] total: 1.26250 | rot_acc: 0.2020

--------------------------------------------------

Saved BEST checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_004.ptPretrain Epoch 5/10: 100%|██████████| 198/198 [11:59<00:00, 3.63s/it, con=0.8782, light=0.0094, mae=0.0881, rot=1.4019, total=1.2291]

Validation: 100%|██████████| 50/50 [01:06<00:00, 1.34s/it, rot_acc=0.2424, val_total=1.0884]--------------------------------------------------

[Epoch 5 TRAIN] total: 1.22906 | mae: 0.08807 | rot: 1.40190 | cont: 0.87818 | light: 0.00940

[Epoch 5 VALID] total: 1.08842 | rot_acc: 0.2424

--------------------------------------------------

Saved BEST checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_005.ptPretrain Epoch 6/10: 100%|██████████| 198/198 [12:01<00:00, 3.64s/it, con=0.6506, light=0.0095, mae=0.0792, rot=1.4412, total=1.1261]

Validation: 100%|██████████| 50/50 [01:07<00:00, 1.34s/it, rot_acc=0.3232, val_total=1.0441]--------------------------------------------------

[Epoch 6 TRAIN] total: 1.12606 | mae: 0.07924 | rot: 1.44117 | cont: 0.65058 | light: 0.00951

[Epoch 6 VALID] total: 1.04405 | rot_acc: 0.3232

--------------------------------------------------

Saved BEST checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_BEST.pt

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_006.ptPretrain Epoch 7/10: 100%|██████████| 198/198 [11:59<00:00, 3.64s/it, con=0.5146, light=0.0092, mae=0.0718, rot=1.4159, total=1.0380]

Validation: 100%|██████████| 50/50 [01:05<00:00, 1.32s/it, rot_acc=0.2020, val_total=1.0637]--------------------------------------------------

[Epoch 7 TRAIN] total: 1.03796 | mae: 0.07179 | rot: 1.41588 | cont: 0.51461 | light: 0.00918

[Epoch 7 VALID] total: 1.06375 | rot_acc: 0.2020

--------------------------------------------------

Saved latest checkpoint: /content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrain_epoch_007.ptPretrain Epoch 8/10: 100%|██████████| 198/198 [11:56<00:00, 3.62s/it, con=0.5213, light=0.0080, mae=0.0659, rot=1.4410, total=1.0478]

Validation: 100%|██████████| 50/50 [01:06<00:00, 1.33s/it, rot_acc=0.3232, val_total=0.9770]--------------------------------------------------

[Epoch 8 TRAIN] total: 1.04782 | mae: 0.06591 | rot: 1.44097 | cont: 0.52125 | light: 0.00796

[Epoch 8 VALID] total: 0.97698 | rot_acc: 0.3232

--------------------------------------------------

# Segmentation

In [ ]:
!pip install -q "monai[all]" torch torchvision nibabel tqdm matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import shutil

from monai.data import Dataset, DataLoader, PersistentDataset, partition_dataset, decollate_batch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandCropByPosNegLabeld, RandFlipd, RandAffined, RandGaussianNoised, RandStdShiftIntensityd, RandBiasFieldd,
    EnsureTyped, AsDiscrete, Activations, RandScaleIntensityd, RandBiasFieldd
)
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.engines import SupervisedTrainer, SupervisedEvaluator # For MONAI Trainer version
from monai.handlers import ( # For MONAI Trainer version
    ValidationHandler, CheckpointSaver, LrScheduleHandler, StatsHandler,
    TensorBoardStatsHandler, MeanDice
)

from tqdm import tqdm
import gc

# --- 0. Configuration & Setup ---
set_determinism(seed=42)

# Paths (Ensure these are correct for your Drive)
image_dir_ft = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/volumes"
label_dir_ft = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/seg"
cache_dir_ft = "/content/drive/MyDrive/Διπλωματικη/cache_finetune"
output_dir_ft_custom = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom" # Output dir for custom loop
output_dir_ft_trainer = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_trainer" # Output dir for MONAI trainer
encoder_weights_path = "/content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrained_swinViT_encoder_weights_BEST.pt"
log_dir_tb = os.path.join(output_dir_ft_trainer, "logs_tb")
# Ensure directories exist
os.makedirs(cache_dir_ft, exist_ok=True)
os.makedirs(output_dir_ft_custom, exist_ok=True)
os.makedirs(output_dir_ft_trainer, exist_ok=True)

# Hyperparameters
ROI_SIZE_FT = (96, 96, 96)
BATCH_SIZE_FT = 2
NUM_EPOCHS_FT = 100
LEARNING_RATE_FT = 1e-4
WEIGHT_DECAY_FT = 1e-5
VAL_INTERVAL = 1
BEST_VAL_METRIC = -1
NUM_WORKERS = 4

# Model Parameters
IN_CHANNELS = 1
OUT_CHANNELS_SEG = 1 # Binary Segmentation
FEATURE_SIZE = 48
USE_CHECKPOINTING = True

# Data Keys
image_key = "image"
label_key = "label"
all_keys = [image_key, label_key]

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- 1. Prepare File Dictionaries & Split Data ---
image_files_ft = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_ft = sorted(glob(os.path.join(label_dir_ft, "*.nii.gz")))

# Basic check and pairing
if len(image_files_ft) != len(label_files_ft):
    print(f"Warning: Mismatch in image/label counts!")
    min_len = min(len(image_files_ft), len(label_files_ft))
    image_files_ft, label_files_ft = image_files_ft[:min_len], label_files_ft[:min_len]

data_dicts_ft = [
    {image_key: img_path, label_key: label_path}
    for img_path, label_path in zip(image_files_ft, label_files_ft)
]
print(f"Found {len(data_dicts_ft)} paired image/label scans.")

train_files_ft, rest_files = partition_dataset(data_dicts_ft, ratios=[0.7, 0.3], shuffle=True, seed=42)
val_files_ft, test_files_ft = partition_dataset(rest_files, ratios=[0.5, 0.5], shuffle=False, seed=42)
print(f"Split: {len(train_files_ft)} train, {len(val_files_ft)} val, {len(test_files_ft)} test.")

# --- 2. Define Transforms ---
train_transforms_ft = Compose([
    LoadImaged(keys=all_keys),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(keys=all_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=all_keys, label_key=label_key, spatial_size=ROI_SIZE_FT,
        pos=1.0, neg=1.0, num_samples=2, image_key=image_key, image_threshold=0.0
    ),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=2),
    RandAffined(
        keys=all_keys, prob=0.2, rotate_range=(0.05, 0.05, 0.05),
        scale_range=(0.1, 0.1, 0.1), mode=("bilinear", "nearest"), padding_mode="zeros"
    ),
    RandGaussianNoised(keys=image_key, prob=0.2, mean=0.0, std=0.01),
    RandStdShiftIntensityd(keys=image_key, factors=0.1, prob=0.3), # Shift intensity based on std dev
    RandScaleIntensityd(keys=image_key, factors=0.1, prob=0.3),    # Scale intensity
    RandBiasFieldd(keys=image_key, prob=0.3),                      # Add bias field artifact

    EnsureTyped(keys=all_keys, dtype=torch.float32),
])

val_transforms_ft = Compose([
    LoadImaged(keys=all_keys),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(keys=all_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=all_keys, dtype=torch.float32),
])

# --- 3. Create Datasets and DataLoaders ---
train_cache_path = os.path.join(cache_dir_ft, "train_seg")
val_cache_path = os.path.join(cache_dir_ft, "val_seg")

train_ds_ft = PersistentDataset(data=train_files_ft, transform=train_transforms_ft, cache_dir=train_cache_path)
val_ds_ft = PersistentDataset(data=val_files_ft, transform=val_transforms_ft, cache_dir=val_cache_path)

train_loader_ft = DataLoader(train_ds_ft, batch_size=BATCH_SIZE_FT, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader_ft = DataLoader(val_ds_ft, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"Created DataLoaders. Train batches: {len(train_loader_ft)}, Val batches: {len(val_loader_ft)}")

# --- 4. Loss, Metric, Post-processing ---
loss_function_seg = DiceCELoss(include_background=False, sigmoid=True)
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

# Post-processing for binary case (Sigmoid activation + thresholding)
post_pred = Compose([EnsureTyped(keys="pred"), Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([EnsureTyped(keys="label"), AsDiscrete(to_onehot=None)]) # Label is already binary

# --- 5. Model Definition Function (to create fresh model for each pipeline) ---
def create_segmentation_model(pretrained_path=None):
    model = SwinUNETR(

        in_channels=IN_CHANNELS,
        out_channels=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        use_checkpoint=USE_CHECKPOINTING
    ).to(DEVICE)

    if pretrained_path and os.path.exists(pretrained_path):
        print(f"Loading pretrained SwinViT encoder weights from: {pretrained_path}")
        try:
            encoder_state_dict = torch.load(pretrained_path, map_location=DEVICE)
            swin_vit_state_dict = {k: v for k, v in encoder_state_dict.items() if k in model.swinViT.state_dict()}
            msg = model.swinViT.load_state_dict(swin_vit_state_dict, strict=False)
            print(f"Pretrained weights loading message: {msg}")
        except Exception as e:
            print(f"Error loading encoder weights: {e}. Training from scratch.")
    else:
        print(f"WARNING: Pretrained weights path not found or not provided. Training from scratch.")
    return model

In [ ]:
# Define cache path for test set if using PersistentDataset
test_cache_path = os.path.join(cache_dir_ft, "test_seg")

# Use PersistentDataset for consistency and potential speedup during evaluation
test_ds_ft = PersistentDataset(data=test_files_ft, transform=val_transforms_ft, cache_dir=test_cache_path)

test_loader_ft = DataLoader(
    test_ds_ft,
    batch_size=1, # Process one volume at a time for evaluation
    shuffle=False, # No need to shuffle test data
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print(f"Created Test DataLoader. Test batches: {len(test_loader_ft)}")

# Segmentation training and Validation Loop SWINUnetR pretrained


In [ ]:

import pandas as pd

cache_dir_ft = "/content/drive/MyDrive/Διπλωματικη/cache_finetune_seg_custom"
output_dir_ft = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom"
encoder_weights_path = "/content/drive/MyDrive/Διπλωματικη/runs/ssl/pretrained_swinViT_encoder_weights_BEST.pt"
history_csv_path = os.path.join(output_dir_ft, "training_history.csv")
model = SwinUNETR(
    # img_size=ROI_SIZE_FT, # Not needed
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS_SEG, # Should be 1
    feature_size=FEATURE_SIZE,
    use_checkpoint=USE_CHECKPOINTING
).to(DEVICE)

# Load Pretrained Weights
if os.path.exists(encoder_weights_path):
    print(f"Loading pretrained SwinViT encoder weights from: {encoder_weights_path}")
    try:
        encoder_state_dict = torch.load(encoder_weights_path, map_location=DEVICE)
        msg = model.swinViT.load_state_dict(encoder_state_dict, strict=False)
        print(f"Pretrained weights loading message: {msg}")
        if set(msg.missing_keys) == set(model.swinViT.state_dict().keys()):
             print("WARNING: No matching keys found! Encoder weights NOT loaded.")
        else:
             print("Pretrained encoder weights loaded.")
    except Exception as e:
        print(f"Error loading encoder weights: {e}. Training from scratch.")
else:
    print(f"WARNING: Pretrained weights not found. Training from scratch.")

# %% Loss, Metric, Post-processing, Optimizer, Scheduler (Same as before)
loss_function = DiceCELoss(include_background=False, sigmoid=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE_FT, weight_decay=WEIGHT_DECAY_FT)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_FT)

# Post-processing for validation metric
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)]) # Label is already binary 0/1

# Metric Definition
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

print("Loss, Optimizer, Scheduler, Metric defined.")

# %% Manual Training & Validation Loop
scaler = GradScaler()
best_val_metric = BEST_VAL_METRIC
start_epoch = 0
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_dice": []
}

latest_checkpoint_path = os.path.join(output_dir_ft_custom, "latest_model_custom.pt")

if os.path.exists(latest_checkpoint_path):
    print(f"Loading checkpoint from: {latest_checkpoint_path}")
    checkpoint = torch.load(latest_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    #if checkpoint.get('scheduler_state'):
        #scheduler.load_state_dict(checkpoint['scheduler_state'])


    print(f"Resetting optimizer LR to: {LEARNING_RATE_FT}")
    for param_group in optimizer.param_groups:
        param_group['lr'] = LEARNING_RATE_FT


    print("Scheduler state NOT loaded from checkpoint.")
    start_epoch = checkpoint['epoch']
    best_val_metric = checkpoint.get('best_val_metric', -1.0)
    print(f"Resuming training from epoch {start_epoch + 1}, previous best metric: {best_val_metric:.4f}")

    history_csv_path = os.path.join(output_dir_ft, "training_history.csv")
    if os.path.exists(history_csv_path):
        print(f"Loading previous history from: {history_csv_path}")
        try:
            df_history_old = pd.read_csv(history_csv_path)
            # Filter out epochs >= start_epoch to avoid duplicates if script restarted mid-epoch
            df_history_old = df_history_old[df_history_old['epoch'] < start_epoch]
            history = df_history_old.to_dict('list')
            print(f"Resumed history up to epoch {history['epoch'][-1] if history['epoch'] else 'N/A'}")
        except Exception as e:
            print(f"Could not load or parse previous history file: {e}. Starting history fresh.")
            history = {"epoch": [], "train_loss": [], "val_loss": [], "val_dice": []}

print(f"\n--- Starting Custom Fine-tuning Loop from epoch {start_epoch + 1} ---")


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_FT)
print(f"Scheduler re-initialized with T_max={NUM_EPOCHS_FT}.")

if start_epoch > 0:
    print(f"Stepping scheduler to epoch {start_epoch}...")
    # Cannot directly set epoch, step it 'start_epoch' times
    # Note: CosineAnnealingLR's state depends only on the last_epoch counter
    # We can directly set this internal counter, but it's simpler to step if few epochs
    # For large start_epoch, directly setting might be better, but requires accessing _step_count
    # Simple approach for moderate epochs:
    for _ in range(start_epoch):
        scheduler.step()
    print(f"Scheduler stepped. Current LR: {optimizer.param_groups[0]['lr']:.6f}")
val_metric_history = []


for epoch in range(start_epoch, NUM_EPOCHS_FT):
    current_epoch = epoch + 1
    print(f"\n===== Epoch {current_epoch}/{NUM_EPOCHS_FT} =====")
    model.train()
    epoch_loss = 0
    step = 0
    pbar_train = tqdm(train_loader_ft, desc=f"Training Epoch {current_epoch}")

    for batch_data in pbar_train:
        step += 1
        # MONAI DataLoader + RandCropByPosNegLabeld(num_samples=2) + BATCH_SIZE_FT=2
        # might yield batch['image'] of shape [4, 1, 96, 96, 96]. Verify this.
        inputs = batch_data[image_key].to(DEVICE)
        labels = batch_data[label_key].to(DEVICE)

        optimizer.zero_grad(set_to_none=True) # More efficient zeroing

        with autocast():
            outputs = model(inputs)
            loss = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer) # Unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip gradients
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar_train.set_postfix(train_loss=f"{loss.item():.4f}")
        del inputs, labels, outputs, loss # Manual cleanup
        if DEVICE == 'cuda': torch.cuda.empty_cache() # Clear cache if needed

    avg_epoch_loss = epoch_loss / step
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {current_epoch} Average Training Loss: {avg_epoch_loss:.4f}, LR: {current_lr:.6f}")

    history["epoch"].append(current_epoch)
    history["train_loss"].append(avg_epoch_loss)

    mean_dice = np.nan
    mean_val_loss = np.nan

    # --- Validation ---
    if current_epoch % VAL_INTERVAL == 0:
        model.eval()
        dice_metric.reset()
        pbar_val = tqdm(val_loader_ft, desc=f"Validation Epoch {current_epoch}")
        with torch.no_grad():
            for val_data in pbar_val:
                val_inputs = val_data[image_key].to(DEVICE)
                val_labels = val_data[label_key].to(DEVICE)

                # Use MONAI's sliding window inference function
                val_outputs = sliding_window_inference(
                    val_inputs, ROI_SIZE_FT, sw_batch_size=2, # Adjust sw_batch_size based on VRAM
                    predictor=model, overlap=0.5, mode="gaussian", device=DEVICE, progress=False
                )

                # Apply postprocessing (on the single item in the batch)
                # Need to decollate because sliding window output might still be batched (B=1)
                val_outputs_post = [post_pred(i) for i in decollate_batch(val_outputs)]
                val_labels_post = [post_label(i) for i in decollate_batch(val_labels)]

                # Compute metric
                dice_metric(y_pred=val_outputs_post, y=val_labels_post)

                pbar_val.set_postfix(current_avg_dice=f"{dice_metric.aggregate().item():.4f}")
                del val_inputs, val_labels, val_outputs, val_outputs_post, val_labels_post # Manual cleanup
                if DEVICE == 'cuda': torch.cuda.empty_cache()

            mean_dice = dice_metric.aggregate().item()
            dice_metric.reset()
            val_metric_history.append(mean_dice)

            print(f"Epoch {current_epoch} Average Validation Dice: {mean_dice:.4f}")

            # --- Checkpoint Saving ---
            checkpoint = {
                'epoch': current_epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_val_metric': best_val_metric, # Store current best for resuming
                'scheduler_state': scheduler.state_dict() if scheduler else None,
                'scaler_state': scaler.state_dict() # Save scaler state
            }

            if mean_dice > best_val_metric:
                best_val_metric = mean_dice
                checkpoint['best_val_metric'] = best_val_metric # Update best metric in dict
                save_path_best = os.path.join(output_dir_ft, "best_metric_model_custom.pt")
                torch.save(checkpoint, save_path_best)
                print(f"✅ Saved new best model with Dice: {best_val_metric:.4f} at {save_path_best}")

            save_path_latest = os.path.join(output_dir_ft, "latest_model_custom.pt")
            torch.save(checkpoint, save_path_latest)
            # print(f"Saved latest checkpoint at {save_path_latest}")

    try:
        epoch_idx = history["epoch"].index(current_epoch)
        # Pad lists if necessary before assigning (ensures correct length)
        while len(history["val_loss"]) <= epoch_idx: history["val_loss"].append(np.nan)
        while len(history["val_dice"]) <= epoch_idx: history["val_dice"].append(np.nan)
        # Update/Assign values for the current epoch's index
        history["val_loss"][epoch_idx] = mean_val_loss
        history["val_dice"][epoch_idx] = mean_dice
    except ValueError:
        print(f"Error: Could not find epoch {current_epoch} in history dict during update. History might be corrupted.")

    # --- Save History AFTER EVERY EPOCH --- # <--- MOVED HERE
    try:
        df_history = pd.DataFrame(history)
        df_history.to_csv(history_csv_path, index=False)
        # print(f"History updated: {history_csv_path}") # Optional print
    except Exception as e:
        print(f"Error saving history to CSV after epoch {current_epoch}: {e}")
    # --- END MOVE ---

    if scheduler:
        scheduler.step()




print(f"🏁 Custom Fine-tuning finished. Best Validation Dice: {best_val_metric:.4f}")

plt.figure("Training & Validation Curves", figsize=(12, 5))

# Plot Training Loss
plt.subplot(1, 2, 1)
plt.title("Epoch Average Training Loss")
if history['train_loss']:
    plt.plot(history['epoch'], history['train_loss'], marker='o', linestyle='-', color="red", label='Avg Train Loss')
    plt.xlabel("Epoch")
    plt.ylabel("DiceCE Loss")
    plt.grid(True)
    plt.legend()
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Plot Validation Loss and Dice
ax_val_loss = plt.subplot(1, 2, 2)
ax_val_dice = ax_val_loss.twinx() # Share x-axis
ax_val_loss.set_title("Validation Metrics per Epoch")

if history['val_loss']:
    # Filter out NaNs for plotting lines, but keep epochs aligned
    val_epochs_with_data = [e for e, vl in zip(history['epoch'], history['val_loss']) if not np.isnan(vl)]
    val_losses_with_data = [vl for vl in history['val_loss'] if not np.isnan(vl)]
    val_dices_with_data = [vd for vd in history['val_dice'] if not np.isnan(vd)] # Assumes NaNs align

    p1, = ax_val_loss.plot(val_epochs_with_data, val_losses_with_data, marker='x', linestyle='--', color="orange", label='Avg Val Loss')
    ax_val_loss.set_xlabel("Epoch")
    ax_val_loss.set_ylabel("DiceCE Loss", color="orange")
    ax_val_loss.tick_params(axis='y', labelcolor="orange")
    ax_val_loss.grid(True, linestyle=':')
    ax_val_loss.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

if history['val_dice']:
    p2, = ax_val_dice.plot(val_epochs_with_data, val_dices_with_data, marker='s', linestyle='-', color="green", label='Avg Val Dice')
    ax_val_dice.set_ylabel("Dice Score", color="green")
    ax_val_dice.tick_params(axis='y', labelcolor="green")
    ax_val_dice.set_ylim(0, max(1.0, max(val_dices_with_data)*1.1) if val_dices_with_data else 1.0)

# Add combined legend
lines = [p1, p2]
ax_val_loss.legend(lines, [l.get_label() for l in lines], loc='center right')


plt.tight_layout()
plt.show()


# %% Testing Phase (Manual Loop - Similar to MONAI Trainer version's test phase)

print("\n--- Starting Testing Phase (Custom Loop) ---")

# Load the best model saved during validation
best_model_path = os.path.join(output_dir_ft, "best_metric_model_custom.pt")
if not os.path.exists(best_model_path):
    print(f"Error: Best model checkpoint not found at {best_model_path}. Skipping testing.")
else:
    print(f"Loading best model from: {best_model_path}")
    # Create a new model instance for testing
    model_test = SwinUNETR(

        in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False # Turn off checkpointing for inference
    ).to(DEVICE)
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model_test.load_state_dict(checkpoint['model_state'])
    model_test.eval()

    # --- Test Dataset and DataLoader ---
    test_transforms = val_transforms_ft
    test_ds = PersistentDataset(data=test_files_ft, transform=test_transforms, cache_dir=os.path.join(cache_dir_ft, "test_cust"))
    test_loader = MonaiDataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

    # --- Test Metric and Post-processing ---
    test_dice_metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
    test_post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
    test_post_label = Compose([AsDiscrete(to_onehot=None)])

    # --- Run Inference and Evaluation ---
    test_dice_vals = []
    pbar_test = tqdm(test_loader_ft, desc="Testing")
    with torch.no_grad():
      with autocast():
        for test_data in pbar_test:
            test_inputs = test_data[image_key].to(DEVICE)
            test_labels = test_data[label_key].to(DEVICE)

            test_outputs = sliding_window_inference(
                test_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model_test, overlap=0.5, mode="gaussian", device=DEVICE, progress=False
            )

            test_outputs_list = [test_post_pred(i) for i in decollate_batch(test_outputs)]
            test_labels_list = [test_post_label(i) for i in decollate_batch(test_labels)]

            test_dice_metric(y_pred=test_outputs_list, y=test_labels_list)
            batch_dice = test_dice_metric.aggregate()
            test_dice_vals.extend(batch_dice.cpu().numpy())
            test_dice_metric.reset()

            pbar_test.set_postfix(mean_dice=f"{np.mean(test_dice_vals):.4f}")
            del test_inputs, test_labels, test_outputs, test_outputs_list, test_labels_list
            if DEVICE == 'cuda': torch.cuda.empty_cache()

    # --- Aggregate and Print Test Results ---
    mean_test_dice = np.mean(test_dice_vals)
    std_test_dice = np.std(test_dice_vals)
    print("\n--- Testing Finished (Custom Loop) ---")
    print(f"Mean Dice on Test Set: {mean_test_dice:.4f} (+/- {std_test_dice:.4f})")
    print(f"Number of test samples evaluated: {len(test_dice_vals)}")

# Polishing step

In [ ]:
# %% --- Polishing Phase ---
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
import os
from tqdm import tqdm
import gc
import matplotlib.pyplot as plt

print("\n--- STARTING POLISHING PHASE ---")

# --- Polishing Configuration ---
POLISH_LR = 1e-6           # A very small, constant learning rate
POLISH_EPOCHS = 10        # Number of extra epochs to train for
# --- End Configuration ---

# --- Define Paths for Polishing ---
# Use the *same* output directory as the main run
output_dir_ft = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom"
# Path to the BEST model from the *previous* training run
base_best_model_path = os.path.join(output_dir_ft, "best_metric_model_custom.pt")
# Path to the LATEST checkpoint from the *previous* run (to get scaler/optimizer)
base_latest_model_path = os.path.join(output_dir_ft, "latest_model_custom.pt") # Use latest to get scaler/optimizer
# Path to the history CSV from the *previous* run
base_history_csv_path = os.path.join(output_dir_ft, "training_history.csv")

# --- Define NEW paths for polishing outputs ---
polish_best_model_path = os.path.join(output_dir_ft, "best_metric_model_polished.pt")
polish_latest_model_path = os.path.join(output_dir_ft, "latest_model_polished.pt")
polish_history_csv_path = os.path.join(output_dir_ft, "training_history_polished.csv")


# --- Initialize Model, Optimizer, Scaler ---
# (Assumes model components like SwinUNETR, loss_function, post_pred, etc. are still in memory)
model = SwinUNETR(
    in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG,
    feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=POLISH_LR, weight_decay=WEIGHT_DECAY_FT) # Use POLISH_LR
scaler = GradScaler()
scheduler = None # No scheduler for polishing
print("Model, Optimizer, and Scaler initialized for polishing.")


# --- ΠΡΟΣΘΕΣΕ ΑΥΤΕΣ ΤΙΣ ΓΡΑΜΜΕΣ ---
loss_function = DiceCELoss(include_background=False, sigmoid=True)
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])
print("Model, Optimizer, Scaler, and Loss/Metric functions initialized for polishing.")
# --- ΤΕΛΟΣ ΠΡΟΣΘΗΚΗΣ ---


# --- Load State from BEST Model Checkpoint ---
if not os.path.exists(base_best_model_path):
    print(f"Error: Best model checkpoint not found at {base_best_model_path}. Cannot start polishing.")
else:
    print(f"Loading best model weights from: {base_best_model_path}")
    checkpoint = torch.load(base_best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])

    # Optionally load optimizer and scaler state from the *latest* checkpoint for continuity
    # (If best model checkpoint doesn't have the latest optimizer state)
    if os.path.exists(base_latest_model_path):
         print(f"Loading optimizer/scaler state from: {base_latest_model_path}")
         latest_checkpoint = torch.load(base_latest_model_path, map_location=DEVICE)
         try:
             optimizer.load_state_dict(latest_checkpoint['optimizer_state'])
             if latest_checkpoint.get('scaler_state'):
                 scaler.load_state_dict(latest_checkpoint['scaler_state'])
             print("Optimizer and Scaler states loaded.")
         except Exception as e:
             print(f"Warning: Could not load optimizer/scaler state, re-initializing. Error: {e}")
             # Re-init optimizer just in case
             optimizer = torch.optim.AdamW(model.parameters(), lr=POLISH_LR, weight_decay=WEIGHT_DECAY_FT)
             scaler = GradScaler()
    else:
        print("Warning: Latest checkpoint not found. Using fresh optimizer/scaler state.")

    # --- Manually Set the Constant Polishing LR ---
    print(f"Setting optimizer LR to constant: {POLISH_LR}")
    for param_group in optimizer.param_groups:
        param_group['lr'] = POLISH_LR

    # --- Load History to Continue Logging ---
    start_epoch = checkpoint.get('epoch', NUM_EPOCHS_FT) # Start from the epoch *after* the best model
    best_val_metric = checkpoint.get('best_val_metric', -1.0) # Start from the previous best
    print(f"Resuming for polishing from epoch {start_epoch + 1}, previous best metric: {best_val_metric:.4f}")

    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_dice": []}
    if os.path.exists(base_history_csv_path):
        print(f"Loading previous history from: {base_history_csv_path}")
        try:
            df_history_old = pd.read_csv(base_history_csv_path)
            # Load the *entire* previous history
            df_history_old = df_history_old[df_history_old['epoch'] <= start_epoch]
            history = df_history_old.to_dict('list')
            for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
                if key not in history: history[key] = []
            print(f"Resumed history up to epoch {history['epoch'][-1] if history['epoch'] else 'N/A'}")
        except Exception as e:
            print(f"Could not load history file: {e}. Starting fresh history.")

    # --- Start Polishing Loop ---
    print(f"\n--- Starting Custom Polishing Loop from epoch {start_epoch + 1} for {POLISH_EPOCHS} epochs ---")

    for epoch in range(start_epoch, start_epoch + POLISH_EPOCHS):
        current_epoch = epoch + 1
        print(f"\n===== Polishing Epoch {current_epoch}/{start_epoch + POLISH_EPOCHS} =====")
        model.train()
        epoch_loss = 0
        step = 0
        pbar_train = tqdm(train_loader_ft, desc=f"Polishing Epoch {current_epoch}")

        for batch_data in pbar_train:
            # --- Training Step (Identical logic) ---
            step += 1; inputs = batch_data[image_key].to(DEVICE); labels = batch_data[label_key].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast(): outputs = model(inputs); loss = loss_function(outputs, labels)
            if not torch.isfinite(loss): print(f"WARNING: Non-finite loss: {loss.item()}. Skipping."); del inputs, labels, outputs, loss; torch.cuda.empty_cache(); continue
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item()
            pbar_train.set_postfix(train_loss=f"{loss.item():.4f}")
            del inputs, labels, outputs, loss; torch.cuda.empty_cache()

        avg_epoch_loss = epoch_loss / step if step > 0 else 0.0
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {current_epoch} Avg Train Loss: {avg_epoch_loss:.4f}, LR: {current_lr:.6f} (Polishing)")

        history["epoch"].append(current_epoch)
        history["train_loss"].append(avg_epoch_loss)
        mean_dice = np.nan; mean_val_loss = np.nan

        # --- Validation (Identical logic) ---
        if current_epoch % VAL_INTERVAL == 0: # Still respects VAL_INTERVAL
            model.eval(); dice_metric.reset(); epoch_val_loss = 0; val_step = 0
            pbar_val = tqdm(val_loader_ft, desc=f"Validation Epoch {current_epoch}")
            with torch.no_grad():
                with autocast():
                    for val_data in pbar_val:
                         val_step += 1; val_inputs = val_data[image_key].to(DEVICE); val_labels = val_data[label_key].to(DEVICE)
                         val_outputs = sliding_window_inference(val_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model, overlap=0.5, mode="gaussian", device=DEVICE, progress=False)
                         v_loss = loss_function(val_outputs, val_labels)
                         if torch.isfinite(v_loss): epoch_val_loss += v_loss.item()
                         else: print(f"WARNING: Non-finite val loss.")
                         val_outputs_post = [post_pred(i) for i in decollate_batch(val_outputs)]
                         val_labels_post = [post_label(i) for i in decollate_batch(val_labels)]
                         dice_metric(y_pred=val_outputs_post, y=val_labels_post)
                         current_avg_loss_val = epoch_val_loss / val_step if val_step > 0 else 0.0
                         pbar_val.set_postfix(avg_dice=f"{dice_metric.aggregate().item():.4f}", avg_loss=f"{current_avg_loss_val:.4f}")
                         del val_inputs, val_labels, val_outputs, val_outputs_post, val_labels_post, v_loss; torch.cuda.empty_cache()

            mean_dice = dice_metric.aggregate().item(); mean_val_loss = epoch_val_loss / val_step if val_step > 0 else np.nan; dice_metric.reset()
            print(f"Epoch {current_epoch} Avg Val Loss: {mean_val_loss:.4f}, Avg Val Dice: {mean_dice:.4f}")

            # --- Checkpoint Saving (Uses NEW paths) ---
            checkpoint = {'epoch': current_epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_metric': best_val_metric, 'scheduler_state': None, 'scaler_state': scaler.state_dict(), 'val_loss': mean_val_loss}
            if mean_dice > best_val_metric:
                best_val_metric = mean_dice
                checkpoint['best_val_metric'] = best_val_metric
                torch.save(checkpoint, polish_best_model_path) # Save to new best path
                print(f"✅ Saved new *polished* best model (Dice: {best_val_metric:.4f}) to {polish_best_model_path}")
            torch.save(checkpoint, polish_latest_model_path) # Save to new latest path

        # --- Append validation metrics (Identical logic) ---
        try:
            epoch_idx = history["epoch"].index(current_epoch);
            while len(history["val_loss"]) <= epoch_idx: history["val_loss"].append(np.nan)
            while len(history["val_dice"]) <= epoch_idx: history["val_dice"].append(np.nan)
            history["val_loss"][epoch_idx] = mean_val_loss; history["val_dice"][epoch_idx] = mean_dice
        except ValueError: print(f"Error: Epoch {current_epoch} not found in history.")

        # --- Save History CSV (Uses NEW path) ---
        try: pd.DataFrame(history).to_csv(polish_history_csv_path, index=False)
        except Exception as e: print(f"Error saving polished history CSV: {e}")

        # --- NO SCHEDULER STEP ---
        # if scheduler: scheduler.step()

    print(f"🏁 Custom Polishing finished. Best Validation Dice: {best_val_metric:.4f}")

    # %% Plotting (Uses the newly populated history dict)
    plt.figure("Polishing Curves", figsize=(12, 5))


In [ ]:

# %% Testing Phase (Manual Loop - Similar to MONAI Trainer version's test phase)

print("\n--- Starting Testing Phase (Custom Loop) ---")

# Load the best model saved during validation
best_model_path = os.path.join(output_dir_ft, "best_metric_model_custom.pt")
if not os.path.exists(best_model_path):
    print(f"Error: Best model checkpoint not found at {best_model_path}. Skipping testing.")
else:
    print(f"Loading best model from: {best_model_path}")
    # Create a new model instance for testing
    model_test = SwinUNETR(

        in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False # Turn off checkpointing for inference
    ).to(DEVICE)
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model_test.load_state_dict(checkpoint['model_state'])
    model_test.eval()

    # --- Test Dataset and DataLoader ---
    test_transforms = val_transforms_ft
    test_ds = PersistentDataset(data=test_files_ft, transform=test_transforms, cache_dir=os.path.join(cache_dir_ft, "test_cust"))
    test_loader_ft = DataLoader(
    test_ds_ft,
    batch_size=1, # Process one volume at a time for evaluation
    shuffle=False, # No need to shuffle test data
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
    )

    # --- Test Metric and Post-processing ---
    test_dice_metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
    test_post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
    test_post_label = Compose([AsDiscrete(to_onehot=None)])

    # --- Run Inference and Evaluation ---
    test_dice_vals = []
    pbar_test = tqdm(test_loader_ft, desc="Testing")
    with torch.no_grad():
      with autocast():
        for test_data in pbar_test:
            test_inputs = test_data[image_key].to(DEVICE)
            test_labels = test_data[label_key].to(DEVICE)

            test_outputs = sliding_window_inference(
                test_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model_test, overlap=0.5, mode="gaussian", device=DEVICE, progress=False
            )

            test_outputs_list = [test_post_pred(i) for i in decollate_batch(test_outputs)]
            test_labels_list = [test_post_label(i) for i in decollate_batch(test_labels)]

            test_dice_metric(y_pred=test_outputs_list, y=test_labels_list)
            batch_dice = test_dice_metric.aggregate()
            test_dice_vals.extend(batch_dice.cpu().numpy())
            test_dice_metric.reset()

            pbar_test.set_postfix(mean_dice=f"{np.mean(test_dice_vals):.4f}")
            del test_inputs, test_labels, test_outputs, test_outputs_list, test_labels_list
            if DEVICE == 'cuda': torch.cuda.empty_cache()

    # --- Aggregate and Print Test Results ---
    mean_test_dice = np.mean(test_dice_vals)
    std_test_dice = np.std(test_dice_vals)
    print("\n--- Testing Finished (Custom Loop) ---")
    print(f"Mean Dice on Test Set: {mean_test_dice:.4f} (+/- {std_test_dice:.4f})")
    print(f"Number of test samples evaluated: {len(test_dice_vals)}")

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator
import os
from glob import glob
import numpy as np
import pandas as pd # Ensure pandas is imported

# --- Configuration ---
# *** Verify this path points to your training history CSV ***
output_dir = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom"
history_csv_path = os.path.join(output_dir, "training_history.csv")


# Function to read history from CSV
def load_history_from_csv(csv_path):
    """Loads training history from a CSV file."""
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_dice": []
    }
    try:
        if os.path.exists(csv_path):
            df_history = pd.read_csv(csv_path)
            history = df_history.to_dict('list')
            print(f"Successfully loaded history from: {csv_path}")
            # Ensure all keys exist even if CSV was partial (though unlikely now)
            for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
                if key not in history: history[key] = [np.nan] * len(history["epoch"])
        else:
            print(f"Warning: History file not found at {csv_path}. Plot will be empty.")
    except Exception as e:
        print(f"Error reading history CSV: {e}")
    return history

# Load the data
history = load_history_from_csv(history_csv_path)

# --- Plotting ---
plt.style.use('seaborn-v0_8-darkgrid')
plt.figure("Training & Validation Curves", figsize=(12, 5))

# --- Subplot 1: Epoch Average Training Loss ---
ax_train = plt.subplot(1, 2, 1)
plt.title("Epoch Average Training Loss")
if history.get('train_loss') and history.get('epoch') and not all(np.isnan(history['train_loss'])):
    train_epochs = history['epoch']
    train_losses = history['train_loss']

    ax_train.plot(train_epochs, train_losses, linestyle='-', color="red", label='Avg Train Loss')
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("DiceCE Loss")
    ax_train.grid(True, linestyle=':')
    ax_train.legend()
    ax_train.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
else:
    ax_train.text(0.5, 0.5, 'Train Loss Data Not Found', ha='center', va='center', fontsize=12)
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("DiceCE Loss")

# --- Subplot 2: Validation Mean Dice ---
ax_val_dice = plt.subplot(1, 2, 2)
plt.title("Average Validation Dice per Epoch")
if history.get('val_dice') and history.get('epoch') and not all(np.isnan(history['val_dice'])):
    # Filter out NaNs for plotting lines (epochs where validation didn't run)
    val_epochs_dice = [e for e, vd in zip(history['epoch'], history['val_dice']) if not np.isnan(vd)]
    val_dices_data = [vd for vd in history['val_dice'] if not np.isnan(vd)]

    if val_epochs_dice: # Only plot if there is valid dice data
        ax_val_dice.plot(val_epochs_dice, val_dices_data, linestyle='-', color="green", label='Avg Val Dice')
        ax_val_dice.set_xlabel("Epoch")
        ax_val_dice.set_ylabel("Dice Score")
        ax_val_dice.set_ylim(0, max(1.0, max(val_dices_data)*1.1) if val_dices_data else 1.0) # Set y-limit
        ax_val_dice.grid(True, linestyle=':')
        ax_val_dice.legend()
        ax_val_dice.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    else:
        ax_val_dice.text(0.5, 0.5, 'Validation Dice Data Not Found', ha='center', va='center', fontsize=12)
        ax_val_dice.set_xlabel("Epoch")
        ax_val_dice.set_ylabel("Dice Score")
else:
     ax_val_dice.text(0.5, 0.5, 'Validation Dice Data Not Found', ha='center', va='center', fontsize=12)
     ax_val_dice.set_xlabel("Epoch")
     ax_val_dice.set_ylabel("Dice Score")

plt.tight_layout()
plt.show()

In [ ]:


import torch
torch.cuda.empty_cache()

In [ ]:
import gc
gc.collect()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os


from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference
from monai.transforms import Activations, AsDiscrete, Compose

# --- Configuration (Ensure these match your training setup) ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROI_SIZE_FT = (96, 96, 96) # The ROI size used during inference/validation
IN_CHANNELS = 1
OUT_CHANNELS_SEG = 1 # Binary Segmentation
FEATURE_SIZE = 48
USE_CHECKPOINTING = True # Keep consistent with training

# Path to the BEST saved model checkpoint
best_model_path = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom/best_metric_model_custom.pt"

# Use the validation or test loader defined previously
# Make sure val_loader_ft or test_loader_ft is defined and accessible
# Example using val_loader_ft:
data_loader_for_viz = val_loader_ft # Or use test_loader_ft

# Number of samples to visualize
num_samples_to_viz = 15

# --- 1. Load the Best Model ---
print(f"Loading best model from: {best_model_path}")
model_viz = SwinUNETR(

    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS_SEG,
    feature_size=FEATURE_SIZE,
    use_checkpoint=USE_CHECKPOINTING
).to(DEVICE)

try:
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    # Handle checkpoints saved by CheckpointSaver (often nested) or directly
    if 'model' in checkpoint: # Saved by CheckpointSaver's save_dict
        model_viz.load_state_dict(checkpoint['model'])
    elif 'model_state' in checkpoint: # Saved manually or by older CheckpointSaver
         model_viz.load_state_dict(checkpoint['model_state'])
    else: # Assume the file *is* the state_dict
        model_viz.load_state_dict(checkpoint)

    model_viz.eval() # Set model to evaluation mode
    print("Best model loaded successfully.")
except FileNotFoundError:
    print(f"Error: Model checkpoint not found at {best_model_path}")
    exit()
except Exception as e:
    print(f"Error loading model state_dict: {e}")
    exit()

# --- 2. Define Post-processing for Visualization ---
# Apply activation (sigmoid) and thresholding
post_pred_viz = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

# --- 3. Iterate Through Loader and Plot ---
with torch.no_grad():
    for i, batch_data in enumerate(data_loader_for_viz):
        if i >= num_samples_to_viz:
            break

        inputs = batch_data["image"].to(DEVICE)
        labels = batch_data["label"].to(DEVICE) # Ground truth mask

        # Perform Inference (using sliding window)
        # Note: sliding_window_inference expects input shape [B, C, H, W, D]
        outputs = sliding_window_inference(
            inputs,
            roi_size=ROI_SIZE_FT,
            sw_batch_size=2, # Adjust based on memory
            predictor=model_viz,
            overlap=0.5,
            mode="gaussian"
        )

        # Apply post-processing to model output
        outputs_processed = post_pred_viz(outputs)

        # Move tensors to CPU and convert to numpy for plotting
        image_np = inputs[0, 0].cpu().numpy() # Take first item in batch, remove channel dim
        label_np = labels[0, 0].cpu().numpy()
        prediction_np = outputs_processed[0, 0].cpu().numpy()

        # Choose a slice index to visualize (e.g., middle slice)
        slice_idx = image_np.shape[2] // 2 # Middle slice along the Depth axis

        print(f"\nVisualizing Sample {i+1} (Slice {slice_idx})")

        plt.figure(f"Sample_{i+1}", figsize=(18, 6))

        # Plot Input Image Slice
        plt.subplot(1, 3, 1)
        plt.title(f"Input Image (Slice {slice_idx})")
        plt.imshow(image_np[:, :, slice_idx], cmap="gray")
        plt.axis("off")

        # Plot Ground Truth Label Slice
        plt.subplot(1, 3, 2)
        plt.title(f"Ground Truth Mask (Slice {slice_idx})")
        plt.imshow(label_np[:, :, slice_idx], cmap="viridis")
        plt.axis("off")

        # Plot Predicted Mask Slice
        plt.subplot(1, 3, 3)
        plt.title(f"Predicted Mask (Slice {slice_idx})")
        plt.imshow(prediction_np[:, :, slice_idx], cmap="viridis")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        # Clean up memory
        del inputs, labels, outputs, outputs_processed, image_np, label_np, prediction_np
        torch.cuda.empty_cache()
        gc.collect()

print("\nVisualization complete.")

Training and Validation on SWINUnetR no pretrain

In [ ]:
# %% Imports
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import shutil
import time
import pandas as pd # Ensure pandas is imported

# MONAI components
from monai.data import DataLoader as MonaiDataLoader, PersistentDataset, partition_dataset, decollate_batch # Use alias if preferred
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandCropByPosNegLabeld, RandFlipd, RandAffined,
    EnsureTyped, AsDiscrete, Activations, RandBiasFieldd, RandStdShiftIntensityd, RandScaleIntensityd, RandGaussianNoised # Added intensity augs
)
from monai.networks.nets import SwinUNETR, UNet # Import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

from tqdm import tqdm
import gc
from IPython.display import display

# %% Configuration & Setup
set_determinism(seed=42)

# --- Base Paths ---
base_drive_path = "/content/drive/MyDrive/Διπλωματικη"
image_dir_ft = os.path.join(base_drive_path, "Brain Tumor Dataset/volumes")
label_dir_ft = os.path.join(base_drive_path, "Brain Tumor Dataset/seg") # Corrected path
cache_base_dir = os.path.join(base_drive_path, "cache_finetune") # Base cache dir for fine-tuning
runs_base_dir = os.path.join(base_drive_path, "runs")
# Path to pretrained weights (only used if load_pretrained=True in a different function)
ssl_weights_path = os.path.join(runs_base_dir, "ssl/pretrained_swinViT_encoder_weights_BEST.pt")

# --- Hyperparameters ---
ROI_SIZE_FT = (96, 96, 96)
BATCH_SIZE_FT = 2
NUM_EPOCHS_FT = 65
LEARNING_RATE_FT = 1e-4
WEIGHT_DECAY_FT = 1e-5
VAL_INTERVAL = 1 # Validate every epoch
NUM_WORKERS = 2 # Reduced based on warning

# --- Model Parameters ---
IN_CHANNELS = 1
OUT_CHANNELS_SEG = 1 # Binary Segmentation
FEATURE_SIZE_SWIN = 48
USE_CHECKPOINTING_SWIN = True

# --- U-Net Specific Parameters ---
UNET_CHANNELS = (16, 32, 64, 128, 256)
UNET_STRIDES = (2, 2, 2, 2)
UNET_DROPOUT = 0.1

# --- Data Keys ---
image_key = "image"
label_key = "label"
all_keys = [image_key, label_key]

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# %% Prepare File Dictionaries & Split Data
image_files_ft = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_ft = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz"))) # Corrected pattern

data_dicts_ft = []
image_basenames = {os.path.basename(f).split('_')[0]: f for f in image_files_ft}
for label_path in label_files_ft:
    label_basename = os.path.basename(label_path)
    patient_id = label_basename.replace("_tumor_segmentation.nii.gz", "")
    corresponding_image = image_basenames.get(patient_id)
    if corresponding_image:
        data_dicts_ft.append({image_key: corresponding_image, label_key: label_path})
    else:
        print(f"Warning: No matching image found for label: {label_basename}")
print(f"Found {len(data_dicts_ft)} paired image/label scans.")

train_files_ft, rest_files = partition_dataset(data_dicts_ft, ratios=[0.7, 0.3], shuffle=True, seed=42)
val_files_ft, test_files_ft = partition_dataset(rest_files, ratios=[0.5, 0.5], shuffle=False, seed=42)
print(f"Split: {len(train_files_ft)} train, {len(val_files_ft)} val, {len(test_files_ft)} test.")

# %% Define Transforms
train_transforms_ft = Compose([
    LoadImaged(keys=all_keys, image_only=False),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(keys=all_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=all_keys, label_key=label_key, spatial_size=ROI_SIZE_FT,
        pos=1.0, neg=1.0, num_samples=2, image_key=image_key, image_threshold=0.0
    ),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=2),
    RandAffined(
        keys=all_keys, prob=0.2, rotate_range=(0.05, 0.05, 0.05),
        scale_range=(0.1, 0.1, 0.1), mode=("bilinear", "nearest"), padding_mode="zeros"
    ),
    RandGaussianNoised(keys=image_key, prob=0.2, mean=0.0, std=0.01),
    RandStdShiftIntensityd(keys=image_key, factors=0.1, prob=0.3), # Shift intensity based on std dev
    RandScaleIntensityd(keys=image_key, factors=0.1, prob=0.3),    # Scale intensity
    RandBiasFieldd(keys=image_key, prob=0.3),                      # Add bias field artifact
    EnsureTyped(keys=all_keys, dtype=torch.float32),
])

val_transforms_ft = Compose([
    LoadImaged(keys=all_keys, image_only=False),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(keys=all_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=all_keys, dtype=torch.float32),
])
test_transforms_ft = val_transforms_ft
print("Transforms defined.")

# %% Create Datasets and DataLoaders (Run Once - These define the shared cache)
os.makedirs(cache_base_dir, exist_ok=True) # Ensure base cache dir exists
train_cache_path = os.path.join(cache_base_dir, "train_seg_shared")
val_cache_path = os.path.join(cache_base_dir, "val_seg_shared")
test_cache_path = os.path.join(cache_base_dir, "test_seg_shared")

# --- Create Datasets using the SHARED cache paths ---
# Clear cache if needed (uncomment if transforms changed significantly)
# if os.path.exists(train_cache_path): shutil.rmtree(train_cache_path)
# if os.path.exists(val_cache_path): shutil.rmtree(val_cache_path)
# if os.path.exists(test_cache_path): shutil.rmtree(test_cache_path)

train_ds_ft = PersistentDataset(data=train_files_ft, transform=train_transforms_ft, cache_dir=train_cache_path)
val_ds_ft = PersistentDataset(data=val_files_ft, transform=val_transforms_ft, cache_dir=val_cache_path)
test_ds_ft = PersistentDataset(data=test_files_ft, transform=test_transforms_ft, cache_dir=test_cache_path)

# --- Create DataLoaders ---
train_loader_ft = MonaiDataLoader(train_ds_ft, batch_size=BATCH_SIZE_FT, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader_ft = MonaiDataLoader(val_ds_ft, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader_ft = MonaiDataLoader(test_ds_ft, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"Created DataLoaders using shared cache. Train: {len(train_loader_ft)}, Val: {len(val_loader_ft)}, Test: {len(test_loader_ft)}")

# %% Loss, Metric, Post-processing (Run Once)
loss_function_seg = DiceCELoss(include_background=False, sigmoid=True)
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])
print("Loss, Metric, Post-processing defined.")

# %% Reusable Training Function (Takes existing DataLoaders, no internal creation or cache logic)
def run_finetuning_scratch(
    model_type: str, # 'swin_unetr_scratch', 'unet_scratch'
    output_dir: str,
    # --- Takes existing loaders ---
    train_loader: MonaiDataLoader,
    val_loader: MonaiDataLoader,
    # --- Pass hyperparameters ---
    num_epochs=NUM_EPOCHS_FT,
    lr=LEARNING_RATE_FT,
    weight_decay=WEIGHT_DECAY_FT,
    val_interval=VAL_INTERVAL,
    device=DEVICE
    ):

    print(f"\n{'='*20} Starting Run: {model_type} {'='*20}")
    print(f"Output directory: {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    # --- DataLoaders are passed in ---
    print(f"Using provided DataLoaders. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    # --- Model Initialization (Inside function, no pretraining) ---
    if model_type == 'swin_unetr_scratch':
        model = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=USE_CHECKPOINTING_SWIN).to(device)
        print(f"SwinUNETR (scratch) initialized.")
    elif model_type == 'unet_scratch':
        model = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(device)
        print(f"UNet (scratch) initialized.")
    else: raise ValueError(f"Unknown model_type: {model_type}")

    # --- Loss, Optimizer, Scheduler, Metric ---
    loss_function = loss_function_seg # Use global
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    # dice_metric, post_pred, post_label assumed global

    # --- Training Loop Setup ---
    scaler = GradScaler()
    best_val_metric = -1.0
    start_epoch = 0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_dice": []}
    history_csv_path = os.path.join(output_dir, f"history_{model_type}.csv")
    latest_checkpoint_path = os.path.join(output_dir, f"latest_model_{model_type}.pt")
    best_checkpoint_path = os.path.join(output_dir, f"best_model_{model_type}.pt")

    # --- Resuming Logic ---
    if os.path.exists(latest_checkpoint_path):
        print(f"Loading checkpoint: {latest_checkpoint_path}")
        checkpoint = torch.load(latest_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        if checkpoint.get('scheduler_state'): scheduler.load_state_dict(checkpoint['scheduler_state'])
        if checkpoint.get('scaler_state'): scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = checkpoint.get('epoch', 0)
        best_val_metric = checkpoint.get('best_val_metric', -1.0)
        print(f"Resuming from epoch {start_epoch + 1}, previous best: {best_val_metric:.4f}")
        if os.path.exists(history_csv_path):
            try:
                df_history_old = pd.read_csv(history_csv_path)
                df_history_old = df_history_old[df_history_old['epoch'] <= start_epoch]
                history = df_history_old.to_dict('list')
                for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
                    if key not in history: history[key] = []
                print(f"Resumed history up to epoch {history['epoch'][-1] if history['epoch'] else 'N/A'}")
            except Exception as e: print(f"Could not load history: {e}. Starting fresh.")
    else:
         print("Starting training from scratch for this model type.")

    # --- Main Loop ---
    print(f"\n--- Starting Loop for {model_type} from epoch {start_epoch + 1} ---")
    for epoch in range(start_epoch, num_epochs):
        current_epoch = epoch + 1
        print(f"\n===== Epoch {current_epoch}/{num_epochs} =====")
        model.train(); epoch_loss = 0; step = 0
        pbar_train = tqdm(train_loader, desc=f"Training {model_type} Epoch {current_epoch}")
        for batch_data in pbar_train:
            step += 1; inputs = batch_data[image_key].to(device); labels = batch_data[label_key].to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(): outputs = model(inputs); loss = loss_function(outputs, labels)
            if not torch.isfinite(loss): print(f"WARNING: Non-finite loss: {loss.item()}. Skipping."); del inputs, labels, outputs, loss; torch.cuda.empty_cache(); continue
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item()
            pbar_train.set_postfix(train_loss=f"{loss.item():.4f}")
            del inputs, labels, outputs, loss; torch.cuda.empty_cache()

        avg_epoch_loss = epoch_loss / step if step > 0 else 0.0; current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {current_epoch} Avg Train Loss: {avg_epoch_loss:.4f}, LR: {current_lr:.6f}")
        history["epoch"].append(current_epoch); history["train_loss"].append(avg_epoch_loss)
        mean_dice = np.nan; mean_val_loss = np.nan

        # --- Validation ---
        if current_epoch % val_interval == 0:
            model.eval(); dice_metric.reset(); epoch_val_loss = 0; val_step = 0
            pbar_val = tqdm(val_loader, desc=f"Validation {model_type} Epoch {current_epoch}")
            with torch.no_grad():
                with autocast():
                    for val_data in pbar_val:
                        val_step += 1; val_inputs = val_data[image_key].to(device); val_labels = val_data[label_key].to(device)
                        val_outputs = sliding_window_inference(val_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model, overlap=0.5, mode="gaussian", device=device, progress=False)
                        v_loss = loss_function(val_outputs, val_labels)
                        if torch.isfinite(v_loss): epoch_val_loss += v_loss.item()
                        else: print(f"WARNING: Non-finite val loss.")
                        val_outputs_post = [post_pred(i) for i in decollate_batch(val_outputs)]
                        val_labels_post = [post_label(i) for i in decollate_batch(val_labels)]
                        dice_metric(y_pred=val_outputs_post, y=val_labels_post)
                        current_avg_loss_val = epoch_val_loss / val_step if val_step > 0 else 0.0
                        pbar_val.set_postfix(avg_dice=f"{dice_metric.aggregate().item():.4f}", avg_loss=f"{current_avg_loss_val:.4f}")
                        del val_inputs, val_labels, val_outputs, val_outputs_post, val_labels_post, v_loss; torch.cuda.empty_cache()

            mean_dice = dice_metric.aggregate().item(); mean_val_loss = epoch_val_loss / val_step if val_step > 0 else np.nan; dice_metric.reset()
            print(f"Epoch {current_epoch} Avg Val Loss: {mean_val_loss:.4f}, Avg Val Dice: {mean_dice:.4f}")
            # --- Checkpoint Saving ---
            checkpoint = {'epoch': current_epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_metric': best_val_metric, 'scheduler_state': scheduler.state_dict() if scheduler else None, 'scaler_state': scaler.state_dict(), 'val_loss': mean_val_loss}
            if mean_dice > best_val_metric:
                best_val_metric = mean_dice; checkpoint['best_val_metric'] = best_val_metric
                torch.save(checkpoint, best_checkpoint_path); print(f"✅ Saved new best model (Dice: {best_val_metric:.4f}) to {best_checkpoint_path}")
            torch.save(checkpoint, latest_checkpoint_path)

        # --- Append validation metrics ---
        try:
            epoch_idx = history["epoch"].index(current_epoch);
            while len(history["val_loss"]) <= epoch_idx: history["val_loss"].append(np.nan)
            while len(history["val_dice"]) <= epoch_idx: history["val_dice"].append(np.nan)
            history["val_loss"][epoch_idx] = mean_val_loss; history["val_dice"][epoch_idx] = mean_dice
        except ValueError: print(f"Error: Epoch {current_epoch} not found in history.")

        # --- Save History CSV ---
        try: pd.DataFrame(history).to_csv(history_csv_path, index=False)
        except Exception as e: print(f"Error saving history CSV: {e}")

        if scheduler: scheduler.step()
    # --- END EPOCH LOOP ---

    print(f"🏁 Finished Training {model_type}. Best Val Dice: {best_val_metric:.4f}")
    return best_val_metric

# %% Function to Run Testing (Takes existing DataLoader, no internal creation or cache logic)
def run_testing(
    model_type: str,
    output_dir: str,
    # --- Takes existing loader ---
    test_loader: MonaiDataLoader # Use alias if preferred
    ):
    print(f"\n--- Starting Testing Phase for {model_type} ---")
    best_model_path = os.path.join(output_dir, f"best_model_{model_type}.pt")
    if not os.path.exists(best_model_path):
        print(f"Error: Best model not found at {best_model_path}. Skipping.")
        return None, None

    print(f"Loading best model: {best_model_path}")
    # Re-initialize model architecture
    if model_type.startswith('swin_unetr'):
        model_test = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=False).to(DEVICE)
    elif model_type == 'unet_scratch':
        model_test = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(DEVICE)
    else: raise ValueError(f"Unknown model_type for testing: {model_type}")

    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model_test.load_state_dict(checkpoint['model_state'])
    model_test.eval()

    # --- Test DataLoader is passed in ---
    print(f"Using provided Test DataLoader. Test batches: {len(test_loader)}")

    # --- Test Metric and Post-processing ---
    test_dice_metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
    test_post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
    test_post_label = Compose([AsDiscrete(to_onehot=None)])

    # --- Run Inference ---
    test_dice_vals = []
    pbar_test = tqdm(test_loader, desc=f"Testing {model_type}")
    with torch.no_grad():
        with autocast():
            for test_data in pbar_test:
                 test_inputs = test_data[image_key].to(DEVICE); test_labels = test_data[label_key].to(DEVICE)
                 test_outputs = sliding_window_inference(test_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model_test, overlap=0.5, mode="gaussian", device=DEVICE, progress=False)
                 test_outputs_list = [test_post_pred(i) for i in decollate_batch(test_outputs)]
                 test_labels_list = [test_post_label(i) for i in decollate_batch(test_labels)]
                 test_dice_metric(y_pred=test_outputs_list, y=test_labels_list)
                 batch_dice = test_dice_metric.aggregate()
                 test_dice_vals.extend(batch_dice.cpu().numpy())
                 test_dice_metric.reset()
                 pbar_test.set_postfix(mean_dice=f"{np.mean(test_dice_vals):.4f}")
                 del test_inputs, test_labels, test_outputs, test_outputs_list, test_labels_list; torch.cuda.empty_cache()

    mean_test_dice = np.mean(test_dice_vals); std_test_dice = np.std(test_dice_vals)
    print(f"\n--- Testing Finished: {model_type} ---")
    print(f"Mean Dice: {mean_test_dice:.4f} (+/- {std_test_dice:.4f})")
    print(f"Num Samples: {len(test_dice_vals)}")
    return mean_test_dice, std_test_dice

# %% Plotting Function (Modified for unique history files)
def plot_histories(model_ids_dict: dict, base_run_dir: str):
    plt.figure("Comparison Curves", figsize=(12, 5))
    colors = plt.cm.viridis(np.linspace(0, 1, len(model_ids_dict) * 2))
    idx = 0

    # Plot Training Loss
    ax_train = plt.subplot(1, 2, 1)
    plt.title("Epoch Average Training Loss")
    for model_id, model_label in model_ids_dict.items():
        history_path = os.path.join(base_run_dir, f"segmentation_{model_id}", f"history_{model_id}.csv")
        if os.path.exists(history_path):
            df = pd.read_csv(history_path)
            ax_train.plot(df['epoch'], df['train_loss'], marker='.', linestyle='-', color=colors[idx], label=f'{model_label}')
            idx += 1
        else:
            print(f"History not found for {model_id} at {history_path}")
    ax_train.set_xlabel("Epoch"); ax_train.set_ylabel("DiceCE Loss"); ax_train.grid(True); ax_train.legend(); ax_train.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    # Plot Validation Dice
    ax_val = plt.subplot(1, 2, 2)
    plt.title("Average Validation Dice")
    idx = len(model_ids_dict) # Reset color index or use separate counter
    for model_id, model_label in model_ids_dict.items():
         history_path = os.path.join(base_run_dir, f"segmentation_{model_id}", f"history_{model_id}.csv")
         if os.path.exists(history_path):
             df = pd.read_csv(history_path)
             df_val = df.dropna(subset=['val_dice']) # Filter NaNs
             ax_val.plot(df_val['epoch'], df_val['val_dice'], marker='.', linestyle='-', color=colors[idx], label=f'{model_label}')
             idx += 1
         else:
             print(f"History not found for {model_id} at {history_path}")
    ax_val.set_xlabel("Epoch"); ax_val.set_ylabel("Dice Score"); ax_val.grid(True); ax_val.legend(); ax_val.set_ylim(0,1); ax_val.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()

# %% Run Experiments
results = {}

# --- Experiment 1: Swin UNETR from Scratch ---
model_id_swin_scratch = 'swin_unetr_scratch'
output_dir_swin_scratch = os.path.join(runs_base_dir, f"segmentation_{model_id_swin_scratch}")
print(f"\n>>> Running Experiment: {model_id_swin_scratch}")
best_val_dice_swin_scratch = run_finetuning_scratch(
    model_type=model_id_swin_scratch,
    output_dir=output_dir_swin_scratch,
    train_loader=train_loader_ft, # Pass existing loader
    val_loader=val_loader_ft     # Pass existing loader
)
test_dice_swin_scratch = run_testing(
    model_id_swin_scratch,
    output_dir_swin_scratch,
    test_loader=test_loader_ft # Pass existing loader
)
if test_dice_swin_scratch and test_dice_swin_scratch[0] is not None:
     results[model_id_swin_scratch] = test_dice_swin_scratch




# --- Compare Results ---
print("\n--- Final Test Results Comparison ---")
if results:
    for model_name, (mean_dice, std_dice) in results.items():
        print(f"{model_name}: Mean Dice = {mean_dice:.4f} +/- {std_dice:.4f}")
else:
    print("No test results recorded.")

# --- Plot Histories ---
model_ids_to_plot = {
     'swin_unetr_scratch': 'SwinUNETR (Scratch)',
     'unet_scratch': 'UNet (Scratch)',
}
# Make sure plot_histories is defined before calling
plot_histories(model_ids_to_plot, runs_base_dir)

In [ ]:
# %% CHECK IN_CHANNELS AUTOMATICALLY
# Φορτώνουμε ένα batch
sample_data = next(iter(train_loader_ft))
image = sample_data[image_key]

print(f"🖼️ Image Shape: {image.shape}")
print(f"✅ YOU SHOULD SET: IN_CHANNELS = {image.shape[1]}")

In [ ]:
# %% Imports
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import shutil
import time
import pandas as pd # Ensure pandas is imported

# MONAI components
from monai.data import DataLoader as MonaiDataLoader, PersistentDataset, partition_dataset, decollate_batch # Use alias if preferred
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandCropByPosNegLabeld, RandFlipd, RandAffined,
    EnsureTyped, AsDiscrete, Activations, RandBiasFieldd, RandStdShiftIntensityd, RandScaleIntensityd, RandGaussianNoised # Added intensity augs
)
from monai.networks.nets import SwinUNETR, UNet # Import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

from tqdm import tqdm
import gc
from IPython.display import display

# %% Configuration & Setup
set_determinism(seed=42)

# --- Base Paths ---
base_drive_path = "/content/drive/MyDrive/Διπλωματικη"
image_dir_ft = os.path.join(base_drive_path, "Brain Tumor Dataset/volumes")
label_dir_ft = os.path.join(base_drive_path, "Brain Tumor Dataset/seg") # Corrected path
cache_base_dir = os.path.join(base_drive_path, "cache_finetune") # Base cache dir for fine-tuning
runs_base_dir = os.path.join(base_drive_path, "runs")
# Path to pretrained weights (only used if load_pretrained=True in a different function)
ssl_weights_path = os.path.join(runs_base_dir, "ssl/pretrained_swinViT_encoder_weights_BEST.pt")

# --- Hyperparameters ---
ROI_SIZE_FT = (96, 96, 96)
BATCH_SIZE_FT = 2
NUM_EPOCHS_FT = 65
LEARNING_RATE_FT = 5e-5
WEIGHT_DECAY_FT = 1e-3
VAL_INTERVAL = 1 # Validate every epoch
NUM_WORKERS = 2 # Reduced based on warning

# --- Model Parameters ---
IN_CHANNELS = 1
OUT_CHANNELS_SEG = 1 # Binary Segmentation
FEATURE_SIZE_SWIN = 48
USE_CHECKPOINTING_SWIN = True

# --- U-Net Specific Parameters ---
UNET_CHANNELS = (16, 32, 64, 128, 256)
UNET_STRIDES = (2, 2, 2, 2)
UNET_DROPOUT = 0.1

# --- Data Keys ---
image_key = "image"
label_key = "label"
all_keys = [image_key, label_key]

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# %% Prepare File Dictionaries & Split Data
image_files_ft = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_ft = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz"))) # Corrected pattern

data_dicts_ft = []
image_basenames = {os.path.basename(f).split('_')[0]: f for f in image_files_ft}
for label_path in label_files_ft:
    label_basename = os.path.basename(label_path)
    patient_id = label_basename.replace("_tumor_segmentation.nii.gz", "")
    corresponding_image = image_basenames.get(patient_id)
    if corresponding_image:
        data_dicts_ft.append({image_key: corresponding_image, label_key: label_path})
    else:
        print(f"Warning: No matching image found for label: {label_basename}")
print(f"Found {len(data_dicts_ft)} paired image/label scans.")

train_files_ft, rest_files = partition_dataset(data_dicts_ft, ratios=[0.7, 0.3], shuffle=True, seed=42)
val_files_ft, test_files_ft = partition_dataset(rest_files, ratios=[0.5, 0.5], shuffle=False, seed=42)
print(f"Split: {len(train_files_ft)} train, {len(val_files_ft)} val, {len(test_files_ft)} test.")

# %% Define Transforms
train_transforms_ft = Compose([
    LoadImaged(keys=all_keys, image_only=False),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(keys=all_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=all_keys, label_key=label_key, spatial_size=ROI_SIZE_FT,
        pos=1.0, neg=1.0, num_samples=2, image_key=image_key, image_threshold=0.0
    ),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=all_keys, prob=0.5, spatial_axis=2),
    RandAffined(
        keys=all_keys, prob=0.2, rotate_range=(0.05, 0.05, 0.05),
        scale_range=(0.1, 0.1, 0.1), mode=("bilinear", "nearest"), padding_mode="zeros"
    ),
    RandGaussianNoised(keys=image_key, prob=0.2, mean=0.0, std=0.01),
    RandStdShiftIntensityd(keys=image_key, factors=0.1, prob=0.3), # Shift intensity based on std dev
    RandScaleIntensityd(keys=image_key, factors=0.1, prob=0.3),    # Scale intensity
    RandBiasFieldd(keys=image_key, prob=0.3),                      # Add bias field artifact
    EnsureTyped(keys=all_keys, dtype=torch.float32),
])

val_transforms_ft = Compose([
    LoadImaged(keys=all_keys, image_only=False),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(keys=all_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=all_keys, dtype=torch.float32),
])
test_transforms_ft = val_transforms_ft
print("Transforms defined.")

# %% Create Datasets and DataLoaders (Run Once - These define the shared cache)
os.makedirs(cache_base_dir, exist_ok=True) # Ensure base cache dir exists
train_cache_path = os.path.join(cache_base_dir, "train_seg_shared")
val_cache_path = os.path.join(cache_base_dir, "val_seg_shared")
test_cache_path = os.path.join(cache_base_dir, "test_seg_shared")

# --- Create Datasets using the SHARED cache paths ---
# Clear cache if needed (uncomment if transforms changed significantly)
# if os.path.exists(train_cache_path): shutil.rmtree(train_cache_path)
# if os.path.exists(val_cache_path): shutil.rmtree(val_cache_path)
# if os.path.exists(test_cache_path): shutil.rmtree(test_cache_path)

train_ds_ft = PersistentDataset(data=train_files_ft, transform=train_transforms_ft, cache_dir=train_cache_path)
val_ds_ft = PersistentDataset(data=val_files_ft, transform=val_transforms_ft, cache_dir=val_cache_path)
test_ds_ft = PersistentDataset(data=test_files_ft, transform=test_transforms_ft, cache_dir=test_cache_path)

# --- Create DataLoaders ---
train_loader_ft = MonaiDataLoader(train_ds_ft, batch_size=BATCH_SIZE_FT, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader_ft = MonaiDataLoader(val_ds_ft, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader_ft = MonaiDataLoader(test_ds_ft, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"Created DataLoaders using shared cache. Train: {len(train_loader_ft)}, Val: {len(val_loader_ft)}, Test: {len(test_loader_ft)}")

# %% Loss, Metric, Post-processing (Run Once)
loss_function_seg = DiceCELoss(include_background=False, sigmoid=True)
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])
print("Loss, Metric, Post-processing defined.")

# %% Reusable Training Function (Takes existing DataLoaders, no internal creation or cache logic)
def run_finetuning_scratch(
    model_type: str, # 'swin_unetr_scratch', 'unet_scratch'
    output_dir: str,
    # --- Takes existing loaders ---
    train_loader: MonaiDataLoader,
    val_loader: MonaiDataLoader,
    # --- Pass hyperparameters ---
    num_epochs=NUM_EPOCHS_FT,
    lr=LEARNING_RATE_FT,
    weight_decay=WEIGHT_DECAY_FT,
    val_interval=VAL_INTERVAL,
    device=DEVICE
    ):

    print(f"\n{'='*20} Starting Run: {model_type} {'='*20}")
    print(f"Output directory: {output_dir}")
    os.makedirs(output_dir, exist_ok=True)

    # --- DataLoaders are passed in ---
    print(f"Using provided DataLoaders. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    # --- Model Initialization (Inside function, no pretraining) ---
    if model_type == 'swin_unetr_scratch':
        model = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=USE_CHECKPOINTING_SWIN).to(device)
        print(f"SwinUNETR (scratch) initialized.")
    elif model_type == 'unet_scratch':
        model = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(device)
        print(f"UNet (scratch) initialized.")

    elif model_type == 'segresnet_scratch':
        model = SegResNet(
            spatial_dims=3,
            in_channels=IN_CHANNELS,
            out_channels=OUT_CHANNELS_SEG,
            init_filters=SEGRESNET_INIT_FILTERS,
            dropout_prob=SEGRESNET_DROPOUT
        ).to(device)
        print(f"SegResNet (scratch) initialized.")
    # >>> END BLOCK <<<
    elif model_type == 'segmamba_scratch':
        from model_segmamba.segmamba import SegMamba
        model = SegMamba(
            in_chans=IN_CHANNELS,
            out_chans=OUT_CHANNELS_SEG,
            spatial_dims=3,
            depths=[2, 2, 2, 2],
            feat_size=[48, 96, 192, 384]
        ).to(DEVICE)
    else: raise ValueError(f"Unknown model_type: {model_type}")

    # --- Loss, Optimizer, Scheduler, Metric ---
    loss_function = loss_function_seg # Use global
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    # dice_metric, post_pred, post_label assumed global

    # --- Training Loop Setup ---
    scaler = GradScaler()
    best_val_metric = -1.0
    start_epoch = 0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_dice": []}
    history_csv_path = os.path.join(output_dir, f"history_{model_type}.csv")
    latest_checkpoint_path = os.path.join(output_dir, f"latest_model_{model_type}.pt")
    best_checkpoint_path = os.path.join(output_dir, f"best_model_{model_type}.pt")

    # --- Resuming Logic ---
    if os.path.exists(latest_checkpoint_path):
        print(f"Loading checkpoint: {latest_checkpoint_path}")
        checkpoint = torch.load(latest_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        if checkpoint.get('scheduler_state'): scheduler.load_state_dict(checkpoint['scheduler_state'])
        if checkpoint.get('scaler_state'): scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = checkpoint.get('epoch', 0)
        best_val_metric = checkpoint.get('best_val_metric', -1.0)
        print(f"Resuming from epoch {start_epoch + 1}, previous best: {best_val_metric:.4f}")
        if os.path.exists(history_csv_path):
            try:
                df_history_old = pd.read_csv(history_csv_path)
                df_history_old = df_history_old[df_history_old['epoch'] <= start_epoch]
                history = df_history_old.to_dict('list')
                for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
                    if key not in history: history[key] = []
                print(f"Resumed history up to epoch {history['epoch'][-1] if history['epoch'] else 'N/A'}")
            except Exception as e: print(f"Could not load history: {e}. Starting fresh.")
    else:
         print("Starting training from scratch for this model type.")

    # --- Main Loop ---
    print(f"\n--- Starting Loop for {model_type} from epoch {start_epoch + 1} ---")
    for epoch in range(start_epoch, num_epochs):
        current_epoch = epoch + 1
        print(f"\n===== Epoch {current_epoch}/{num_epochs} =====")
        model.train(); epoch_loss = 0; step = 0
        pbar_train = tqdm(train_loader, desc=f"Training {model_type} Epoch {current_epoch}")
        for batch_data in pbar_train:
            step += 1; inputs = batch_data[image_key].to(device); labels = batch_data[label_key].to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(): outputs = model(inputs); loss = loss_function(outputs, labels)
            if not torch.isfinite(loss): print(f"WARNING: Non-finite loss: {loss.item()}. Skipping."); del inputs, labels, outputs, loss; torch.cuda.empty_cache(); continue
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item()
            pbar_train.set_postfix(train_loss=f"{loss.item():.4f}")
            del inputs, labels, outputs, loss; torch.cuda.empty_cache()

        avg_epoch_loss = epoch_loss / step if step > 0 else 0.0; current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {current_epoch} Avg Train Loss: {avg_epoch_loss:.4f}, LR: {current_lr:.6f}")
        history["epoch"].append(current_epoch); history["train_loss"].append(avg_epoch_loss)
        mean_dice = np.nan; mean_val_loss = np.nan

        # --- Validation ---
        if current_epoch % val_interval == 0:
            model.eval(); dice_metric.reset(); epoch_val_loss = 0; val_step = 0
            pbar_val = tqdm(val_loader, desc=f"Validation {model_type} Epoch {current_epoch}")
            with torch.no_grad():
                with autocast():
                    for val_data in pbar_val:
                        val_step += 1; val_inputs = val_data[image_key].to(device); val_labels = val_data[label_key].to(device)
                        val_outputs = sliding_window_inference(val_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model, overlap=0.5, mode="gaussian", device=device, progress=False)
                        v_loss = loss_function(val_outputs, val_labels)
                        if torch.isfinite(v_loss): epoch_val_loss += v_loss.item()
                        else: print(f"WARNING: Non-finite val loss.")
                        val_outputs_post = [post_pred(i) for i in decollate_batch(val_outputs)]
                        val_labels_post = [post_label(i) for i in decollate_batch(val_labels)]
                        dice_metric(y_pred=val_outputs_post, y=val_labels_post)
                        current_avg_loss_val = epoch_val_loss / val_step if val_step > 0 else 0.0
                        pbar_val.set_postfix(avg_dice=f"{dice_metric.aggregate().item():.4f}", avg_loss=f"{current_avg_loss_val:.4f}")
                        del val_inputs, val_labels, val_outputs, val_outputs_post, val_labels_post, v_loss; torch.cuda.empty_cache()

            mean_dice = dice_metric.aggregate().item(); mean_val_loss = epoch_val_loss / val_step if val_step > 0 else np.nan; dice_metric.reset()
            print(f"Epoch {current_epoch} Avg Val Loss: {mean_val_loss:.4f}, Avg Val Dice: {mean_dice:.4f}")
            # --- Checkpoint Saving ---
            checkpoint = {'epoch': current_epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_metric': best_val_metric, 'scheduler_state': scheduler.state_dict() if scheduler else None, 'scaler_state': scaler.state_dict(), 'val_loss': mean_val_loss}
            if mean_dice > best_val_metric:
                best_val_metric = mean_dice; checkpoint['best_val_metric'] = best_val_metric
                torch.save(checkpoint, best_checkpoint_path); print(f"✅ Saved new best model (Dice: {best_val_metric:.4f}) to {best_checkpoint_path}")
            torch.save(checkpoint, latest_checkpoint_path)

        # --- Append validation metrics ---
        try:
            epoch_idx = history["epoch"].index(current_epoch);
            while len(history["val_loss"]) <= epoch_idx: history["val_loss"].append(np.nan)
            while len(history["val_dice"]) <= epoch_idx: history["val_dice"].append(np.nan)
            history["val_loss"][epoch_idx] = mean_val_loss; history["val_dice"][epoch_idx] = mean_dice
        except ValueError: print(f"Error: Epoch {current_epoch} not found in history.")

        # --- Save History CSV ---
        try: pd.DataFrame(history).to_csv(history_csv_path, index=False)
        except Exception as e: print(f"Error saving history CSV: {e}")

        if scheduler: scheduler.step()
    # --- END EPOCH LOOP ---

    print(f"🏁 Finished Training {model_type}. Best Val Dice: {best_val_metric:.4f}")
    return best_val_metric

# %% Function to Run Testing (Takes existing DataLoader, no internal creation or cache logic)
def run_testing(
    model_type: str,
    output_dir: str,
    # --- Takes existing loader ---
    test_loader: MonaiDataLoader # Use alias if preferred
    ):
    print(f"\n--- Starting Testing Phase for {model_type} ---")
    best_model_path = os.path.join(output_dir, f"best_model_{model_type}.pt")
    if not os.path.exists(best_model_path):
        print(f"Error: Best model not found at {best_model_path}. Skipping.")
        return None, None

    print(f"Loading best model: {best_model_path}")
    # Re-initialize model architecture
    if model_type.startswith('swin_unetr'):
        model_test = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=False).to(DEVICE)
    elif model_type == 'unet_scratch':
        model_test = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(DEVICE)
    elif model_type == 'segresnet_scratch':
        model_test = SegResNet(
            spatial_dims=3,
            in_channels=IN_CHANNELS,
            out_channels=OUT_CHANNELS_SEG,
            init_filters=SEGRESNET_INIT_FILTERS,
            dropout_prob=SEGRESNET_DROPOUT
        ).to(DEVICE)
    elif model_type == 'segmamba_scratch':
        from model_segmamba.segmamba import SegMamba
        model_test = SegMamba(
            in_chans=IN_CHANNELS,
            out_chans=OUT_CHANNELS_SEG,
            spatial_dims=3,
            depths=[2, 2, 2, 2],
            feat_size=[48, 96, 192, 384]
        ).to(DEVICE)
    else: raise ValueError(f"Unknown model_type for testing: {model_type}")

    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model_test.load_state_dict(checkpoint['model_state'])
    model_test.eval()

    # --- Test DataLoader is passed in ---
    print(f"Using provided Test DataLoader. Test batches: {len(test_loader)}")

    # --- Test Metric and Post-processing ---
    test_dice_metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
    test_post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
    test_post_label = Compose([AsDiscrete(to_onehot=None)])

    # --- Run Inference ---
    test_dice_vals = []
    pbar_test = tqdm(test_loader, desc=f"Testing {model_type}")
    with torch.no_grad():
        with autocast():
            for test_data in pbar_test:
                 test_inputs = test_data[image_key].to(DEVICE); test_labels = test_data[label_key].to(DEVICE)
                 test_outputs = sliding_window_inference(test_inputs, ROI_SIZE_FT, sw_batch_size=2, predictor=model_test, overlap=0.5, mode="gaussian", device=DEVICE, progress=False)
                 test_outputs_list = [test_post_pred(i) for i in decollate_batch(test_outputs)]
                 test_labels_list = [test_post_label(i) for i in decollate_batch(test_labels)]
                 test_dice_metric(y_pred=test_outputs_list, y=test_labels_list)
                 batch_dice = test_dice_metric.aggregate()
                 test_dice_vals.extend(batch_dice.cpu().numpy())
                 test_dice_metric.reset()
                 pbar_test.set_postfix(mean_dice=f"{np.mean(test_dice_vals):.4f}")
                 del test_inputs, test_labels, test_outputs, test_outputs_list, test_labels_list; torch.cuda.empty_cache()

    mean_test_dice = np.mean(test_dice_vals); std_test_dice = np.std(test_dice_vals)
    print(f"\n--- Testing Finished: {model_type} ---")
    print(f"Mean Dice: {mean_test_dice:.4f} (+/- {std_test_dice:.4f})")
    print(f"Num Samples: {len(test_dice_vals)}")
    return mean_test_dice, std_test_dice

# %% Plotting Function (Modified for unique history files)
def plot_histories(model_ids_dict: dict, base_run_dir: str):
    plt.figure("Comparison Curves", figsize=(12, 5))
    colors = plt.cm.viridis(np.linspace(0, 1, len(model_ids_dict) * 2))
    idx = 0

    # Plot Training Loss
    ax_train = plt.subplot(1, 2, 1)
    plt.title("Epoch Average Training Loss")
    for model_id, model_label in model_ids_dict.items():
        history_path = os.path.join(base_run_dir, f"segmentation_{model_id}", f"history_{model_id}.csv")
        if os.path.exists(history_path):
            df = pd.read_csv(history_path)
            ax_train.plot(df['epoch'], df['train_loss'], marker='.', linestyle='-', color=colors[idx], label=f'{model_label}')
            idx += 1
        else:
            print(f"History not found for {model_id} at {history_path}")
    ax_train.set_xlabel("Epoch"); ax_train.set_ylabel("DiceCE Loss"); ax_train.grid(True); ax_train.legend(); ax_train.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    # Plot Validation Dice
    ax_val = plt.subplot(1, 2, 2)
    plt.title("Average Validation Dice")
    idx = len(model_ids_dict) # Reset color index or use separate counter
    for model_id, model_label in model_ids_dict.items():
         history_path = os.path.join(base_run_dir, f"segmentation_{model_id}", f"history_{model_id}.csv")
         if os.path.exists(history_path):
             df = pd.read_csv(history_path)
             df_val = df.dropna(subset=['val_dice']) # Filter NaNs
             ax_val.plot(df_val['epoch'], df_val['val_dice'], marker='.', linestyle='-', color=colors[idx], label=f'{model_label}')
             idx += 1
         else:
             print(f"History not found for {model_id} at {history_path}")
    ax_val.set_xlabel("Epoch"); ax_val.set_ylabel("Dice Score"); ax_val.grid(True); ax_val.legend(); ax_val.set_ylim(0,1); ax_val.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()

# %% Run Experiments
results = {}

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from monai.networks.nets import SegResNet
# 1. Define Parameters
SEGRESNET_INIT_FILTERS = 16
SEGRESNET_DROPOUT = 0.2

In [ ]:
 # --- Experiment 3: SegResNet from Scratch ---
from monai.networks.nets import SegResNet

# 1. Define Parameters
SEGRESNET_INIT_FILTERS = 16
SEGRESNET_DROPOUT = 0.2
model_id_segresnet = 'segresnet_scratch'  # Matches the key we added to the functions
output_dir_segresnet = os.path.join(runs_base_dir, f"segmentation_{model_id_segresnet}")

print(f"\n>>> Running Experiment: {model_id_segresnet}")

# 2. Run Training
best_val_dice_segresnet = run_finetuning_scratch(
    model_type=model_id_segresnet,
    output_dir=output_dir_segresnet,
    train_loader=train_loader_ft,
    val_loader=val_loader_ft
)

# 3. Run Testing
test_dice_segresnet = run_testing(
    model_type=model_id_segresnet,
    output_dir=output_dir_segresnet,
    test_loader=test_loader_ft
)

# 4. Save Results
if test_dice_segresnet and test_dice_segresnet[0] is not None:
     results[model_id_segresnet] = test_dice_segresnet

print(f"\n✅ Experiment {model_id_segresnet} Completed.")

In [ ]:
# --- Experiment 2: UNet from Scratch ---
model_id_unet_scratch = 'unet_scratch'
output_dir_unet_scratch = os.path.join(runs_base_dir, f"segmentation_{model_id_unet_scratch}")
print(f"\n>>> Running Experiment: {model_id_unet_scratch}")
best_val_dice_unet_scratch = run_finetuning_scratch(
    model_type=model_id_unet_scratch,
    output_dir=output_dir_unet_scratch,
    train_loader=train_loader_ft, # Pass existing loader
    val_loader=val_loader_ft     # Pass existing loader
)
test_dice_unet_scratch = run_testing(
    model_id_unet_scratch,
    output_dir_unet_scratch,
    test_loader=test_loader_ft # Pass existing loader
)
if test_dice_unet_scratch and test_dice_unet_scratch[0] is not None:
     results[model_id_unet_scratch] = test_dice_unet_scratch


## MAMBA


In [ ]:
!git clone https://github.com/ge-xing/SegMamba.git



In [ ]:
!cd SegMamba

In [ ]:
import transformers
print(f"🕵️ Current Transformers Version: {transformers.__version__}")

In [ ]:
from transformers.generation import GreedySearchDecoderOnlyOutput
print("✅ Success: 'GreedySearchDecoderOnlyOutput' found!")

In [ ]:
!pip uninstall -y transformers

# 2. Εγκατάσταση της "χρυσής" έκδοσης για SegMamba
!pip install transformers==4.39.3

In [ ]:
# %% CORRECT INSTALLATION (Using bundled source)
import os

# 1. Καθαρισμός από προηγούμενες (λάθος) εγκαταστάσεις
print("🗑️ Uninstalling incorrect versions...")
!pip uninstall -y mamba-ssm causal-conv1d

# 2. Εγκατάσταση του bundled Causal-Conv1d
# Μπαίνουμε στον φάκελο του repo και χτίζουμε το δικό τους πακέτο
print("⏳ Installing bundled Causal-Conv1d...")
if os.path.exists("SegMamba/causal-conv1d"):
    !cd SegMamba/causal-conv1d && pip install .
else:
    print("❌ Error: Folder SegMamba/causal-conv1d not found!")

# 3. Εγκατάσταση του bundled Mamba
# Αυτό είναι το κρίσιμο βήμα που έδειξε το link σου
print("⏳ Installing bundled Mamba (Takes ~5 mins)...")
if os.path.exists("SegMamba/mamba"):
    !cd SegMamba/mamba && pip install .
else:
    print("❌ Error: Folder SegMamba/mamba not found!")

print("✅ Installation complete. Now try importing.")

In [ ]:
# %% Verify SegMamba Import
import sys
import os
import torch

# Προσθήκη path
repo_path = os.path.abspath("SegMamba")
if repo_path not in sys.path:
    sys.path.append(repo_path)

try:
    from model_segmamba.segmamba import SegMamba

    # Δοκιμή στο GPU
    model = SegMamba(
        in_chans=1,
        out_chans=1,
        depths=[2, 2, 2, 2],
        feat_size=[48, 96, 192, 384]
    ).to('cuda')

    print("✅ SegMamba initialized successfully on GPU!")
    del model

except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 TIP: If it still fails, go to 'Runtime' -> 'Restart Session' and try ONLY this cell again.")

In [ ]:
# %% TEST IMPORT
import sys
import os
import torch

repo_path = os.path.abspath("SegMamba")
if repo_path not in sys.path:
    sys.path.append(repo_path)

try:
    # Τώρα θα πρέπει να δουλέψει
    from model_segmamba.segmamba import SegMamba

    print("🔄 Testing SegMamba on GPU...")
    model = SegMamba(
        in_chans=1,
        out_chans=1,
        depths=[2, 2, 2, 2],
        feat_size=[48, 96, 192, 384]
    ).to('cuda')

    print("✅ SUCCESS! SegMamba is ready.")
    del model

except ImportError as e:
    print(f"❌ Import Error: {e}")
except Exception as e:
    print(f"❌ Runtime Error: {e}")

In [ ]:
# %% Experiment 4: SegMamba from Scratch
model_id_segmamba = 'segmamba_scratch'
output_dir_segmamba = os.path.join(runs_base_dir, f"segmentation_{model_id_segmamba}")

print(f"\n>>> Running Experiment: {model_id_segmamba}")

# 1. Train
best_val_dice_segmamba = run_finetuning_scratch(
    model_type=model_id_segmamba,
    output_dir=output_dir_segmamba,
    train_loader=train_loader_ft,
    val_loader=val_loader_ft
)

# 2. Test
test_dice_segmamba = run_testing(
    model_type=model_id_segmamba,
    output_dir=output_dir_segmamba,
    test_loader=test_loader_ft
)

# 3. Save Results
if test_dice_segmamba and test_dice_segmamba[0] is not None:
     results['SegMamba'] = test_dice_segmamba

print("✅ SegMamba Experiment Completed.")

In [ ]:


import pandas as pd
import os
import shutil
from glob import glob
from tqdm import tqdm # For progress bar
import re

# 1. Paths (MODIFY THESE)
csv_path = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/balanced_metadata_gradeaware.csv" # Path to your CSV file with patient IDs
original_volume_dir = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/volumes" # Path to ALL original volumes
original_segmentation_dir = "/content/drive/MyDrive/Διπλωματικη/Brain Tumor Dataset/seg" # Path to ALL original segmentations

# Where to create the new subset folders
subset_base_dir = "/content/drive/MyDrive/Διπλωματικη/Brain_Tumor_Subset"
subset_volume_dir = os.path.join(subset_base_dir, "volumes_subset")
subset_segmentation_dir = os.path.join(subset_base_dir, "segmentations_subset")

# 2. CSV Column Name
patient_id_column = "ID" # The exact name of the column with IDs in your CSV

# --- End Configuration ---

# --- Script Logic ---

# Create subset directories if they don't exist
os.makedirs(subset_volume_dir, exist_ok=True)
os.makedirs(subset_segmentation_dir, exist_ok=True)
print(f"Subset directories created/ensured at: {subset_base_dir}")

# Read patient IDs from CSV
try:
    df_subset = pd.read_csv(csv_path)
    if patient_id_column not in df_subset.columns:
        raise ValueError(f"Column '{patient_id_column}' not found in CSV file: {csv_path}")
    subset_ids = df_subset[patient_id_column].astype(str).tolist() # Ensure IDs are strings
    print(f"Read {len(subset_ids)} patient IDs from {csv_path}")
except FileNotFoundError:
    print(f"Error: CSV file not found at {csv_path}")
    exit()
except ValueError as ve:
    print(f"Error: {ve}")
    exit()
except Exception as e:
    print(f"An error occurred reading the CSV: {e}")
    exit()

# Counters for tracking
volume_copy_count = 0
segmentation_copy_count = 0
missing_files_log = []

# --- Find and Copy Files ---
print("Searching for and copying files...")

# Iterate through the desired patient IDs from the CSV
for patient_id_csv in tqdm(subset_ids, desc="Processing Patients"):
    # --- Format the ID to match filenames (e.g., UCSF-PDGM-170 -> UCSF-PDGM-0170) ---
    match = re.match(r"([A-Za-z-]+-)(\d+)", patient_id_csv)
    if match:
        prefix = match.group(1) # e.g., "UCSF-PDGM-"
        number_part = match.group(2) # e.g., "170"
        # Format number to 4 digits with leading zeros
        formatted_number = number_part.zfill(4) # e.g., "0170"
        patient_id_formatted = f"{prefix}{formatted_number}" # e.g., "UCSF-PDGM-0170"
    else:
        # If the ID doesn't match the expected pattern, log it and skip
        missing_files_log.append(f"Skipping ID: '{patient_id_csv}' - does not match expected pattern (e.g., PREFIX-NUMBER)")
        continue # Skip to the next ID in the CSV

    # --- Find Volume Files using the formatted ID ---
    volume_pattern = os.path.join(original_volume_dir, f"{patient_id_formatted}_*.nii.gz") # <-- Use formatted ID
    found_volumes = glob(volume_pattern)

    if not found_volumes:
        missing_files_log.append(f"Volume(s) not found for formatted ID: {patient_id_formatted} (from CSV ID: {patient_id_csv})")
    else:
        for vol_file_path in found_volumes:
            filename = os.path.basename(vol_file_path)
            destination_path = os.path.join(subset_volume_dir, filename)
            try:
                shutil.copy2(vol_file_path, destination_path)
                volume_copy_count += 1
            except Exception as e:
                missing_files_log.append(f"Error copying volume {filename} for ID {patient_id_formatted}: {e}")

    # --- Find Segmentation File using the formatted ID ---
    segmentation_filename = f"{patient_id_formatted}_tumor_segmentation.nii.gz" # <-- Use formatted ID
    segmentation_file_path = os.path.join(original_segmentation_dir, segmentation_filename)

    if not os.path.exists(segmentation_file_path):
        missing_files_log.append(f"Segmentation not found for formatted ID: {patient_id_formatted} (from CSV ID: {patient_id_csv}) (Expected: {segmentation_filename})")
    else:
        destination_path = os.path.join(subset_segmentation_dir, segmentation_filename)
        try:
            shutil.copy2(segmentation_file_path, destination_path)
            segmentation_copy_count += 1
        except Exception as e:
            missing_files_log.append(f"Error copying segmentation {segmentation_filename} for ID {patient_id_formatted}: {e}")

# --- Summary ---
# ... (rest of the summary print statements remain the same) ...
print("\n--- Processing Complete ---")
print(f"Target Patient IDs from CSV: {len(subset_ids)}")
print(f"Volume files copied: {volume_copy_count}")
print(f"Segmentation files copied: {segmentation_copy_count}")

if missing_files_log:
    print("\n--- Issues Encountered ---")
    for log in missing_files_log:
        print(log)
else:
    print("\nAll requested files found and copied successfully.")

In [ ]:
# %% 1. FINAL BENCHMARK FUNCTION (NO POLISHING)
import os
import time
import gc
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch
from monai.metrics import (
    DiceMetric,
    HausdorffDistanceMetric,
    SurfaceDiceMetric,
    ConfusionMatrixMetric,
    MeanIoU
)
from monai.transforms import Compose, Activations, AsDiscrete

# Transforms
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

def run_benchmark_standard(
    model_id: str,
    folder_name: str,
    test_loader,
    roi_size=(96, 96, 96)
):
    print(f"\n{'='*10} 🧪 BENCHMARKING: {model_id} {'='*10}")

    # --- Path Handling ---
    output_dir = os.path.join(runs_base_dir, folder_name)
    best_model_path = os.path.join(output_dir, f"best_model_{model_id}.pt")

    # Ειδική περίπτωση για το Pretrained SwinUNETR που έχει custom όνομα
    if not os.path.exists(best_model_path):
        alt_path = os.path.join(output_dir, "best_metric_model_custom.pt")
        if os.path.exists(alt_path):
            best_model_path = alt_path
            print("🔹 Found custom named checkpoint.")

    if not os.path.exists(best_model_path):
        print(f"❌ Error: Model file not found at {best_model_path}")
        return None

    # --- A. Model Initialization ---
    print(f"🏗️ Initializing Architecture...")
    try:
        if 'swin' in model_id:
            model = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=False).to(DEVICE)
        elif 'mamba' in model_id:
            from model_segmamba.segmamba import SegMamba
            model = SegMamba(in_chans=IN_CHANNELS, out_chans=OUT_CHANNELS_SEG, depths=[2, 2, 2, 2], feat_size=[48, 96, 192, 384]).to(DEVICE)
        elif 'segresnet' in model_id:
            model = SegResNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, init_filters=SEGRESNET_INIT_FILTERS, dropout_prob=SEGRESNET_DROPOUT).to(DEVICE)
        elif 'unet' in model_id:
            model = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(DEVICE)
        else:
            raise ValueError(f"Unknown model_id: {model_id}")
    except Exception as e:
        print(f"❌ Init Error: {e}")
        return None

    # --- B. Load Weights ---
    print(f"📥 Loading weights...")
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    # --- C. Metrics Setup ---
    dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
    iou_metric = MeanIoU(include_background=False, reduction="mean", get_not_nans=False)
    hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean", get_not_nans=False)
    nsd_metric = SurfaceDiceMetric(include_background=False, class_thresholds=[1.0], reduction="mean", get_not_nans=False)
    conf_matrix = ConfusionMatrixMetric(include_background=False, metric_name=["sensitivity", "precision", "accuracy"], reduction="mean", get_not_nans=False)

    inference_times = []

    # --- D. Inference Loop ---
    with torch.no_grad():
        for test_data in tqdm(test_loader, desc=f"Testing {model_id}"):
            t_in = test_data[image_key].to(DEVICE)
            t_lbl = test_data[label_key].to(DEVICE)

            with torch.cuda.amp.autocast(): # ή torch.amp.autocast('cuda')
                # Timing
                start_time = time.time()
                t_out = sliding_window_inference(t_in, roi_size, 2, model, overlap=0.5)
                end_time = time.time()
                inference_times.append(end_time - start_time)

                # Post-processing
                val_outputs = [post_pred(i) for i in decollate_batch(t_out)]
                val_labels = [post_label(i) for i in decollate_batch(t_lbl)]

                # Update Metrics
                dice_metric(y_pred=val_outputs, y=val_labels)
                iou_metric(y_pred=val_outputs, y=val_labels)
                hd95_metric(y_pred=val_outputs, y=val_labels)
                nsd_metric(y_pred=val_outputs, y=val_labels)
                conf_matrix(y_pred=val_outputs, y=val_labels)

    # --- E. Aggregation ---
    results = {}
    results['Model'] = model_id
    results['Dice'] = dice_metric.aggregate().item()
    results['IoU'] = iou_metric.aggregate().item()
    results['HD95'] = hd95_metric.aggregate().item()
    results['NSD'] = nsd_metric.aggregate().item()

    conf_res = conf_matrix.aggregate()
    results['Sensitivity'] = conf_res[0].item()
    results['Precision'] = conf_res[1].item()
    results['Accuracy'] = conf_res[2].item()
    results['Time_Sample_sec'] = np.mean(inference_times)

    # --- F. Cleanup ---
    print("🧹 Cleaning up memory...")
    del model, dice_metric, iou_metric, hd95_metric, nsd_metric, conf_matrix
    torch.cuda.empty_cache()
    gc.collect()

    print(f"✅ Finished {model_id}: Dice={results['Dice']:.4f}")
    return results

In [ ]:
# %% 2. RUN FINAL LEADERBOARD
# Λίστα με τα μοντέλα που έχεις έτοιμα
benchmark_list = [
    # 1. SegMamba (Ο Νικητής;)
    {"id": "segmamba_scratch", "folder": "segmentation_segmamba_scratch"},

    # 2. SwinUNETR Pretrained (To βαρύ πυροβολικό)
    {"id": "swin_unetr_finetune", "folder": "segmentation_finetune_custom"},

    # 3. SegResNet (O γρήγορος)
    {"id": "segresnet_scratch", "folder": "segmentation_segresnet_scratch"},

    # 4. UNet (Το Baseline)
    {"id": "unet_scratch", "folder": "segmentation_unet_scratch"},

    # 5. SwinUNETR Scratch (Αν υπάρχει)
    {"id": "swin_unetr_scratch", "folder": "segmentation_swin_unetr_scratch"},
]

final_scores = []

print(f"{'='*30}\n🏆 STARTING FINAL THESIS BENCHMARK\n{'='*30}")

for config in benchmark_list:
    try:
        # Χρησιμοποιούμε τον val_loader_ft για όλα, για δίκαιη σύγκριση
        res = run_benchmark_standard(
            model_id=config["id"],
            folder_name=config["folder"],
            test_loader=val_loader_ft,
            roi_size=(96, 96, 96)
        )
        if res:
            final_scores.append(res)

    except Exception as e:
        print(f"⚠️ Could not benchmark {config['id']}: {e}")
        torch.cuda.empty_cache()

# --- Display & Save Results ---
if final_scores:
    df_final = pd.DataFrame(final_scores)

    # Ταξινόμηση και μορφοποίηση
    cols = ['Model', 'Dice', 'IoU', 'HD95', 'NSD', 'Sensitivity', 'Precision', 'Accuracy', 'Time_Sample_sec']
    df_final = df_final[cols].sort_values(by='Dice', ascending=False)

    print("\n\n📄 FINAL RESULTS FOR THESIS:")
    print(df_final.to_markdown(index=False, floatfmt=".4f"))

    save_path = os.path.join(runs_base_dir, "FINAL_THESIS_RESULTS.csv")
    df_final.to_csv(save_path, index=False)
    print(f"\n💾 Saved to: {save_path}")
else:
    print("❌ No results found.")

# Focal refinement

In [ ]:
# %% 1. DEFINITION: POLISHING FUNCTION
from monai.losses import DiceLoss, FocalLoss
from monai.metrics import DiceMetric

def run_polishing_final(model_type, output_dir, train_loader, val_loader, roi_size=(96, 96, 96)):
    print(f"\n{'='*10} ✨ POLISHING: {model_type} {'='*10}")

    # --- Paths ---
    # Ψάχνουμε το υπάρχον best model
    original_best_path = os.path.join(output_dir, f"best_model_{model_type}.pt")
    # Αν δεν βρεθεί με το τυπικό όνομα, ψάχνουμε για το custom όνομα (για το pretrained)
    if not os.path.exists(original_best_path):
        original_best_path = os.path.join(output_dir, "best_metric_model_custom.pt")

    # Το νέο αρχείο που θα σωθεί
    polished_best_path = os.path.join(output_dir, f"best_model_{model_type}_polished.pt")

    if not os.path.exists(original_best_path):
        print(f"❌ Error: Could not find base model at {original_best_path}. Skipping.")
        return

    print(f"📥 Loading base weights from: {original_best_path}")

    # --- 1. Init Architecture ---
    if 'swin' in model_type:
        # Χειρισμός για SwinUNETR (Scratch & Pretrained)
        model = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=False).to(DEVICE)
    elif 'mamba' in model_type:
        from model_segmamba.segmamba import SegMamba
        model = SegMamba(in_chans=IN_CHANNELS, out_chans=OUT_CHANNELS_SEG, spatial_dims=3, depths=[2, 2, 2, 2], feat_size=[48, 96, 192, 384]).to(DEVICE)
    elif 'segresnet' in model_type:
        model = SegResNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, init_filters=SEGRESNET_INIT_FILTERS, dropout_prob=SEGRESNET_DROPOUT).to(DEVICE)
    elif 'unet' in model_type:
        model = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(DEVICE)
    else:
        raise ValueError(f"Unknown model: {model_type}")

    # --- 2. Load Weights ---
    checkpoint = torch.load(original_best_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    current_best_dice = checkpoint.get('best_val_metric', 0.0)
    print(f"   Current Baseline Dice: {current_best_dice:.4f}")

    # --- 3. Setup for Polishing ---
    # Hybrid Loss: Dice + Focal (gamma=2.0 for hard mining)
    loss_dice = DiceLoss(sigmoid=True, batch=True)
    loss_focal = FocalLoss(gamma=2.0)

    # Πολύ χαμηλό Learning Rate (το 1e-5 είναι οριακό, αν δεις πάλι αστάθεια πήγαινε σε 5e-6)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=1e-2)
    scaler = GradScaler()
    dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

    # --- 4. Training Loop (10 Epochs) ---
    for epoch in range(1, 11):
        model.train()
        epoch_loss = 0
        step = 0

        # Training
        for batch_data in train_loader:
            step += 1
            inputs, labels = batch_data[image_key].to(DEVICE), batch_data[label_key].to(DEVICE)

            optimizer.zero_grad()
            with autocast():
                outputs = model(inputs)
                # Συνδυασμένο Loss: 1.0*Dice + 0.5*Focal
                loss = loss_dice(outputs, labels) + 0.5 * loss_focal(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        # Validation
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                v_in, v_lbl = val_data[image_key].to(DEVICE), val_data[label_key].to(DEVICE)
                with autocast():
                    v_out = sliding_window_inference(v_in, roi_size, 2, model, overlap=0.5)
                    val_outputs = [post_pred(i) for i in decollate_batch(v_out)]
                    val_labels = [post_label(i) for i in decollate_batch(v_lbl)]
                    dice_metric(y_pred=val_outputs, y=val_labels)

        val_dice = dice_metric.aggregate().item()
        dice_metric.reset()

        print(f"   Ep {epoch}/10 | Loss: {epoch_loss/step:.4f} | Val Dice: {val_dice:.4f}")

        # Save ONLY if it improves over the ORIGINAL best
        if val_dice > current_best_dice:
            current_best_dice = val_dice
            torch.save({'model_state': model.state_dict(), 'best_val_metric': val_dice}, polished_best_path)
            print(f"   🏆 IMPROVED! New Dice: {val_dice:.4f} -> Saved to {polished_best_path}")

    # Αν δεν βελτιώθηκε ποτέ, σώζουμε το τελευταίο checkpoint ως polished (για να έχουμε αρχείο)
    if not os.path.exists(polished_best_path):
        print("   ⚠️ No improvement. Saving original weights as polished for consistency.")
        torch.save({'model_state': model.state_dict(), 'best_val_metric': current_best_dice}, polished_best_path)
    # 6. MEMORY CLEANUP (ΚΡΙΣΙΜΟ) 🧹
    print(f"🧹 Cleaning up memory for {model_type}...")
    del model
    del optimizer
    del scaler
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ Polishing Done for {model_type}.")

In [ ]:

torch.cuda.empty_cache()
gc.collect()

In [ ]:
# %% 2. EXECUTE POLISHING ON ALL MODELS
all_models_config = [

    # 2. SegMamba Scratch
    {
        "id": "segmamba_scratch",
        "folder": "segmentation_segmamba_scratch",
        "loader": train_loader_ft,
        "roi": (96, 96, 96)
    },

    # 3. SegResNet Scratch
    {
        "id": "segresnet_scratch",
        "folder": "segmentation_segresnet_scratch",
        "loader": train_loader_ft,
        "roi": (96, 96, 96)
    },

    # 4. SwinUNETR Scratch
    {
        "id": "swin_unetr_scratch",
        "folder": "segmentation_swin_unetr_scratch",
        "loader": train_loader_ft,
        "roi": (96, 96, 96)
    },

    # 5. UNet Scratch
    {
        "id": "unet_scratch",
        "folder": "segmentation_unet_scratch",
        "loader": train_loader_ft,
        "roi": (96, 96, 96)
    }
]

print("🚀 STARTING MASSIVE POLISHING SESSION (5 Models x 10 Epochs)")

for config in all_models_config:
    full_output_dir = os.path.join(runs_base_dir, config["folder"])

    if os.path.exists(full_output_dir):
        run_polishing_final(
            model_type=config["id"],
            output_dir=full_output_dir,
            train_loader=config["loader"],
            val_loader=val_loader_ft, # Κοινός Val Loader
            roi_size=config["roi"]
        )
    else:
        print(f"⚠️ Directory not found for {config['id']}: {full_output_dir}")

In [ ]:
# %% UPDATED run_testing WITH ALL METRICS
from monai.metrics import DiceMetric, HausdorffDistanceMetric, SurfaceDiceMetric, MeanIoU, ConfusionMatrixMetric
import numpy as np
import pandas as pd
import time

def run_testing_final_metrics(
    model_type: str,
    output_dir: str,
    test_loader: MonaiDataLoader, # O Test Loader που έφτιαξες
    roi_size=(96, 96, 96)
    ):
    print(f"\n{'='*10} Final Testing Phase for {model_type} {'='*10}")

    # 1. Load Model Weights
    best_model_path = os.path.join(output_dir, f"best_model_{model_type}.pt")
    # Fallback για το custom όνομα του swin_unetr
    if not os.path.exists(best_model_path) and "swin" in model_type:
         alt_path = os.path.join(output_dir, "best_metric_model_custom.pt")
         if os.path.exists(alt_path): best_model_path = alt_path

    if not os.path.exists(best_model_path):
        print(f"❌ Error: Best model not found at {best_model_path}.")
        return None

    print(f"📥 Loading model: {os.path.basename(best_model_path)}")

    # 2. Re-initialize Architecture
    if model_type.startswith('swin_unetr'):
        model_test = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=False).to(DEVICE)
    elif model_type == 'unet_scratch':
        model_test = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT).to(DEVICE)
    elif model_type == 'segresnet_scratch':
        model_test = SegResNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, init_filters=SEGRESNET_INIT_FILTERS, dropout_prob=SEGRESNET_DROPOUT).to(DEVICE)
    elif model_type == 'segmamba_scratch':
        try:
            from model_segmamba.segmamba import SegMamba
        except ImportError:
            import sys; sys.path.append('/content/drive/MyDrive/Διπλωματικη')
            from model_segmamba.segmamba import SegMamba
        model_test = SegMamba(in_chans=IN_CHANNELS, out_chans=OUT_CHANNELS_SEG, depths=[2, 2, 2, 2], feat_size=[48, 96, 192, 384]).to(DEVICE)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model_test.load_state_dict(checkpoint['model_state'])
    model_test.eval()

    print(f"✅ Using provided Test DataLoader. Batches: {len(test_loader)}")

    # 3. Define Metrics (Same config as your original script where possible)
    # Χρησιμοποιούμε 'mean_batch' όπως εσύ, για να πάρουμε το ίδιο Dice
    dice_metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
    iou_metric = MeanIoU(include_background=False, reduction="mean_batch", get_not_nans=False)
    # Για HD95/NSD το reduction 'mean' είναι το standard
    hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean", get_not_nans=False)
    nsd_metric = SurfaceDiceMetric(include_background=False, class_thresholds=[1.0], reduction="mean", get_not_nans=False)

    # Sensitivity/Precision/Accuracy
    conf_matrix = ConfusionMatrixMetric(include_background=False, metric_name=["sensitivity", "precision", "accuracy"], reduction="mean", get_not_nans=False)

    post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
    post_label = Compose([AsDiscrete(to_onehot=None)])

    # 4. Inference Loop
    dice_scores = []
    inference_times = []

    with torch.no_grad():
        for test_data in tqdm(test_loader, desc=f"Testing {model_type}"):
            inputs = test_data[image_key].to(DEVICE)
            labels = test_data[label_key].to(DEVICE)

            with autocast():
                start_time = time.time()
                outputs = sliding_window_inference(inputs, roi_size, 2, model_test, overlap=0.5, mode="gaussian")
                end_time = time.time()
                inference_times.append(end_time - start_time)

                # Post-processing
                val_outputs = [post_pred(i) for i in decollate_batch(outputs)]
                val_labels = [post_label(i) for i in decollate_batch(labels)]

                # Update Metrics
                dice_metric(y_pred=val_outputs, y=val_labels)

                # Παίρνουμε το Dice αυτού του batch και το αποθηκεύουμε όπως στο δικό σου script
                batch_dice = dice_metric.aggregate()
                dice_scores.extend(batch_dice.cpu().numpy())

                # Υπόλοιπες μετρικές
                iou_metric(y_pred=val_outputs, y=val_labels)
                hd95_metric(y_pred=val_outputs, y=val_labels)
                nsd_metric(y_pred=val_outputs, y=val_labels)
                conf_matrix(y_pred=val_outputs, y=val_labels)

                # Reset Dice only (since we aggregate manually)
                dice_metric.reset()

    # 5. Aggregate Results
    mean_dice = np.mean(dice_scores) # Αυτό θα σου δώσει το 0.8962!
    std_dice = np.std(dice_scores)

    results = {
        "Model": model_type,
        "Dice": mean_dice,
        "Std": std_dice,
        "IoU": iou_metric.aggregate().item(),
        "HD95": hd95_metric.aggregate().item(),
        "NSD": nsd_metric.aggregate().item(),
        "Sensitivity": conf_matrix.aggregate()[0].item(),
        "Precision": conf_matrix.aggregate()[1].item(),
        "Accuracy": conf_matrix.aggregate()[2].item(),
        "Time_Per_Sample": np.mean(inference_times)
    }

    print(f"\n🏁 Finished {model_type}")
    print(f"   🎲 Dice: {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"   📏 HD95: {results['HD95']:.4f}")

    # Cleanup
    del model_test, dice_metric, hd95_metric, nsd_metric, conf_matrix
    torch.cuda.empty_cache()

    return results

In [ ]:

benchmark_config = [
    {"id": "segmamba_scratch",  "folder": "segmentation_segmamba_scratch"},
    {"id": "swin_unetr_finetune", "folder": "segmentation_finetune_custom"},
    {"id": "segresnet_scratch", "folder": "segmentation_segresnet_scratch"},
    {"id": "swin_unetr_scratch", "folder": "segmentation_swin_unetr_scratch"},
    {"id": "unet_scratch",      "folder": "segmentation_unet_scratch"},
]

final_leaderboard = []

print(f"{'='*40}\n🏆 FINAL THESIS BENCHMARK (Running on Test Set)\n{'='*40}")

for config in benchmark_config:
    try:
        # Χρησιμοποιούμε το test_loader_ft που ξέρουμε ότι είναι το σωστό Test split
        res = run_testing_final_metrics(
            model_type=config["id"],
            output_dir=os.path.join(runs_base_dir, config["folder"]),
            test_loader=test_loader_ft,
            roi_size=(96, 96, 96)
        )
        if res:
            final_leaderboard.append(res)
    except Exception as e:
        print(f"⚠️ Error with {config['id']}: {e}")

# Display & Save
if final_leaderboard:
    df = pd.DataFrame(final_leaderboard)
    cols = ['Model', 'Dice', 'Std', 'HD95', 'NSD', 'IoU', 'Sensitivity', 'Time_Per_Sample']
    df = df[cols].sort_values(by='Dice', ascending=False)

    print("\n\n📄 FINAL RESULTS TABLE:")
    print(df.to_markdown(index=False, floatfmt=".4f"))

    df.to_csv(os.path.join(runs_base_dir, "THESIS_FINAL_RESULTS.csv"), index=False)

In [ ]:
# %% ULTIMATE MASSIVE VISUALIZATION SCRIPT
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import sys
from tqdm import tqdm
from monai.inferers import sliding_window_inference
from monai.transforms import Compose, Activations, AsDiscrete
from monai.metrics import DiceMetric

# --- 1. GLOBAL SETTINGS ---
# Ορίζουμε ποιον Loader θα χρησιμοποιήσουμε (Validation ή Test)
VIS_LOADER = val_loader_ft
ROI_SIZE = (96, 96, 96)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Φάκελος εξόδου
SAVE_DIR = os.path.join(runs_base_dir, "THESIS_FINAL_VISUALS")
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📂 All images will be saved to: {SAVE_DIR}")

# Λίστα Μοντέλων προς Οπτικοποίηση
models_to_viz = [
    {"id": "segmamba_scratch",  "folder": "segmentation_segmamba_scratch", "name": "SegMamba"},
    {"id": "swin_unetr_finetune", "folder": "segmentation_finetune_custom", "name": "SwinUNETR (Pre)"},
    {"id": "swin_unetr_scratch", "folder": "segmentation_swin_unetr_scratch", "name": "SwinUNETR (Scratch)"},
    {"id": "segresnet_scratch", "folder": "segmentation_segresnet_scratch", "name": "SegResNet"},
    {"id": "unet_scratch",      "folder": "segmentation_unet_scratch", "name": "UNet Baseline"},
]

# Transforms για υπολογισμό Dice
post_pred_dice = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_dice = Compose([AsDiscrete(to_onehot=None)])
dice_metric_viz = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

# --- 2. HELPER FUNCTION: Load Model Architecture ---
def load_model_arch(model_id):
    if model_id == 'segmamba_scratch':
        try:
            from model_segmamba.segmamba import SegMamba
        except ImportError:
            sys.path.append('/content/drive/MyDrive/Διπλωματικη')
            from model_segmamba.segmamba import SegMamba
        model = SegMamba(in_chans=IN_CHANNELS, out_chans=OUT_CHANNELS_SEG, depths=[2, 2, 2, 2], feat_size=[48, 96, 192, 384])
    elif model_id.startswith('swin'):
        model = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, feature_size=FEATURE_SIZE_SWIN, use_checkpoint=False)
    elif model_id == 'segresnet_scratch':
        model = SegResNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, init_filters=SEGRESNET_INIT_FILTERS, dropout_prob=SEGRESNET_DROPOUT)
    elif model_id == 'unet_scratch':
        model = UNet(spatial_dims=3, in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS_SEG, channels=UNET_CHANNELS, strides=UNET_STRIDES, num_res_units=2, norm="instance", dropout=UNET_DROPOUT)
    else:
        raise ValueError(f"Unknown model id: {model_id}")
    return model.to(DEVICE)

# --- 3. HELPER FUNCTION: Plotting Logic ---
def plot_and_save(model_name, case_type, patient_idx, vol_img, vol_lbl, vol_pred, dice_score):
    # Βρίσκουμε την τομή (slice) με τον μεγαλύτερο αληθινό όγκο
    slice_areas = np.sum(vol_lbl, axis=(0, 1))
    max_slice_idx = np.argmax(slice_areas)
    if np.max(slice_areas) == 0: # Αν είναι κενή μάσκα, παίρνουμε τη μεσαία τομή
        max_slice_idx = vol_lbl.shape[2] // 2

    # Εξαγωγή 2D slices (περιστροφή 90 μοίρες για σωστό προσανατολισμό)
    img_slice = np.rot90(vol_img[:, :, max_slice_idx])
    lbl_slice = np.rot90(vol_lbl[:, :, max_slice_idx])
    pred_slice = np.rot90(vol_pred[:, :, max_slice_idx])

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # MRI
    axes[0].imshow(img_slice, cmap="gray")
    axes[0].set_title(f"Patient Idx: {patient_idx} | MRI T1c", fontsize=12)
    axes[0].axis("off")

    # Ground Truth (Green)
    axes[1].imshow(img_slice, cmap="gray")
    masked_lbl = np.ma.masked_where(lbl_slice == 0, lbl_slice)
    axes[1].imshow(masked_lbl, cmap="spring", alpha=0.7) # Spring=Magenta/Greenish
    axes[1].set_title("Ground Truth (Target)", fontsize=12)
    axes[1].axis("off")

    # Prediction (Red)
    axes[2].imshow(img_slice, cmap="gray")
    masked_pred = np.ma.masked_where(pred_slice == 0, pred_slice)
    axes[2].imshow(masked_pred, cmap="autumn", alpha=0.7) # Autumn=Red/Orange
    axes[2].set_title(f"Prediction | Dice: {dice_score:.4f}", fontsize=12)
    axes[2].axis("off")

    plt.suptitle(f"Model: {model_name} | Case Type: {case_type.upper()}", fontsize=16, y=0.95)
    plt.tight_layout()

    # Save
    safe_model_name = model_name.replace(" ", "_").replace("(", "").replace(")", "")
    filename = os.path.join(SAVE_DIR, f"{safe_model_name}_{case_type}_Idx{patient_idx}_Dice{dice_score:.3f}.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)

# --- 4. MAIN LOOP GENERATOR ---
print(f"{'='*50}\n🚀 STARTING MASSIVE VISUALIZATION GENERATION\n{'='*50}")

for config in models_to_viz:
    model_id = config["id"]
    model_name = config["name"]
    folder = config["folder"]
    print(f"\n📸 Processing Model: {model_name}...")

    # A. Load Weights
    weights_path = os.path.join(runs_base_dir, folder, f"best_model_{model_id}.pt")
    if not os.path.exists(weights_path) and "swin" in model_id:
         alt = os.path.join(runs_base_dir, folder, "best_metric_model_custom.pt")
         if os.path.exists(alt): weights_path = alt

    if not os.path.exists(weights_path):
        print(f"❌ Weights not found for {model_id}. Skipping.")
        continue

    try:
        model = load_model_arch(model_id)
        ckpt = torch.load(weights_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        model.eval()
    except Exception as e:
        print(f"❌ Error loading {model_id}: {e}. Skipping.")
        continue

    # B. PASS 1: Scan for Scores (Memory Efficient)
    print("   🔍 Pass 1: Scanning dataset for scores...")
    patient_scores = []
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(VIS_LOADER, desc=f"Scanning {model_name}", leave=False)):
            inputs, labels = batch_data[image_key].to(DEVICE), batch_data[label_key].to(DEVICE)
            with torch.cuda.amp.autocast():
                outputs = sliding_window_inference(inputs, ROI_SIZE, 2, model, overlap=0.5)
                val_out = post_pred_dice(outputs[0])
                val_lbl = post_label_dice(labels[0])
                dice_metric_viz(y_pred=val_out.unsqueeze(0), y=val_lbl.unsqueeze(0))
                score = dice_metric_viz.aggregate().item()
                dice_metric_viz.reset()
                patient_scores.append({'idx': i, 'dice': score})

    # C. Select Indices
    patient_scores.sort(key=lambda x: x['dice'])
    n = len(patient_scores)
    targets = [
        {'type': 'Worst',  'data': patient_scores[0]},
        {'type': 'Median', 'data': patient_scores[n//2]},
        {'type': 'Best',   'data': patient_scores[-1]}
    ]
    target_indices = [t['data']['idx'] for t in targets]
    print(f"   🎯 Selected indices: Worst={targets[0]['data']['idx']} (Dice {targets[0]['data']['dice']:.3f}), Median={targets[1]['data']['idx']} (Dice {targets[1]['data']['dice']:.3f}), Best={targets[2]['data']['idx']} (Dice {targets[2]['data']['dice']:.3f})")

    # D. PASS 2: Targeted Inference & Plotting
    print("   🎨 Pass 2: Generating plots for selected cases...")
    plots_done = 0
    with torch.no_grad():
         for i, batch_data in enumerate(tqdm(VIS_LOADER, desc=f"Plotting {model_name}", leave=False)):
             if i in target_indices:
                 # Βρίσκουμε ποιος τύπος είναι (Best/Median/Worst)
                 target_info = next(t for t in targets if t['data']['idx'] == i)

                 inputs = batch_data[image_key].to(DEVICE)
                 labels = batch_data[label_key].to(DEVICE)
                 with torch.cuda.amp.autocast():
                     outputs = sliding_window_inference(inputs, ROI_SIZE, 2, model, overlap=0.5, mode="gaussian")
                     # Εδώ χρειαζόμαστε το raw sigmoid output για την οπτικοποίηση, όχι το discrete
                     pred_sigmoid = torch.sigmoid(outputs[0]).cpu().numpy()[0]
                     # Εφαρμόζουμε threshold 0.5 για την τελική μάσκα
                     pred_mask = (pred_sigmoid > 0.5).astype(np.float32)

                     img_np = inputs[0, 0].cpu().numpy()
                     lbl_np = labels[0, 0].cpu().numpy()

                     plot_and_save(model_name, target_info['type'], i, img_np, lbl_np, pred_mask, target_info['data']['dice'])
                     plots_done += 1
             if plots_done == 3: break # Τελειώσαμε για αυτό το μοντέλο

    # Cleanup memory
    del model
    torch.cuda.empty_cache()
    print(f"✅ Finished {model_name}.")

print(f"\n{'='*50}\n🎉 DONE! Check the folder: {SAVE_DIR}\n{'='*50}")

In [ ]:
# %% 3D VISUALIZATION SCRIPT
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure
import numpy as np
import os

def plot_3d_volumetric(vol_lbl, vol_pred, title, save_path):
    """
    Δημιουργεί 3D αναπαράσταση Ground Truth vs Prediction.
    """
    # 1. Έλεγχος αν υπάρχουν όγκοι (για να μην σκάσει ο marching_cubes)
    if np.sum(vol_lbl) < 10 or np.sum(vol_pred) < 10:
        print(f"⚠️ Skipping 3D plot for {title} (Tumor too small or empty).")
        return

    # 2. Ρύθμιση του Plot
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    # --- GROUND TRUTH (Πράσινο - Solid/Transparent) ---
    try:
        # Marching Cubes: Μετατροπή voxels σε τρίγωνα (Mesh)
        verts, faces, _, _ = measure.marching_cubes(vol_lbl, level=0.5)
        mesh = Poly3DCollection(verts[faces], alpha=0.2) # Alpha=0.2 για διαφάνεια
        mesh.set_facecolor('green')
        ax.add_collection3d(mesh)
    except Exception as e:
        print(f"Error meshing GT: {e}")

    # --- PREDICTION (Κόκκινο - Wireframe/Solid) ---
    try:
        verts_p, faces_p, _, _ = measure.marching_cubes(vol_pred, level=0.5)
        mesh_p = Poly3DCollection(verts_p[faces_p], alpha=0.3)
        mesh_p.set_facecolor('red')
        # mesh_p.set_edgecolor('k') # Αν θες περίγραμμα
        ax.add_collection3d(mesh_p)
    except Exception as e:
        print(f"Error meshing Pred: {e}")

    # 4. Καλλωπισμός (Limits & Angles)
    ax.set_xlim(0, vol_lbl.shape[0])
    ax.set_ylim(0, vol_lbl.shape[1])
    ax.set_zlim(0, vol_lbl.shape[2])

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title, fontsize=15)

    # Αλλαγή γωνίας θέασης για καλύτερη προοπτική
    ax.view_init(elev=30, azim=45)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ Saved 3D render: {save_path}")

# --- ΠΑΡΑΔΕΙΓΜΑ ΧΡΗΣΗΣ (Τρέξε αυτό αφού τρέξεις το προηγούμενο Visualization Loop) ---
# Θα πάρουμε τα δεδομένα του 'Median' case που βρήκαμε στο προηγούμενο βήμα
# Προϋπόθεση: Να έχεις τρέξει το "ULTIMATE VISUALIZATION SCRIPT" και να έχεις τα targets

# Αν τα targets υπάρχουν στη μνήμη από το προηγούμενο κελί:
if 'targets' in locals() and 'model' in locals():
    # Παίρνουμε τον Median Case
    median_idx = targets[1]['data']['idx']
    dice_val = targets[1]['data']['dice']

    print(f"\nGeneraring 3D Plot for Median Case (Idx {median_idx})...")

    # Ξανα-φορτώνουμε τα δεδομένα του Median case
    # (Εδώ κάνουμε ένα mini-loop για να τον βρούμε, λίγο μπακάλικο αλλά ασφαλές)
    found = False
    for i, batch_data in enumerate(VIS_LOADER):
        if i == median_idx:
            inputs = batch_data[image_key].to(DEVICE)
            labels = batch_data[label_key].to(DEVICE)

            with torch.no_grad():
                with torch.cuda.amp.autocast():
                    outputs = sliding_window_inference(inputs, ROI_SIZE, 2, model, overlap=0.5, mode="gaussian")
                    pred_mask = (torch.sigmoid(outputs[0]) > 0.5).float().cpu().numpy()[0]
                    lbl_mask = labels[0, 0].cpu().numpy()

            # Καλουμε τη συνάρτηση 3D
            save_name = os.path.join(SAVE_DIR, f"3D_{model_name}_Median_Dice{dice_val:.3f}.png")
            plot_3d_volumetric(lbl_mask, pred_mask, f"{model_name} Median Case (Green=GT, Red=Pred)", save_name)
            found = True
            break

    if not found: print("Could not find the index in loader.")
else:
    print("⚠️ Πρώτα τρέξε το 'ULTIMATE VISUALIZATION SCRIPT' για να γεμίσουν οι μεταβλητές!")

In [ ]:
import random
import torch
import matplotlib.pyplot as plt
import numpy as np
from monai.inferers import sliding_window_inference

# --- 1. SET TARGETS ---
fixed_targets = [65, 73]
num_random = 5
all_indices = list(range(len(VIS_LOADER.dataset)))
remaining_indices = [i for i in all_indices if i not in fixed_targets]
random_targets = random.sample(remaining_indices, num_random)
final_target_indices = fixed_targets + random_targets

print(f"🎯 Target Indices: {fixed_targets} | 🎲 Random Indices: {random_targets}")

# --- 2. EXECUTION LOOP ---
for config in models_to_viz:
    model_id = config["id"]
    model_name = config["name"]
    folder = config["folder"]

    weights_path = os.path.join(runs_base_dir, folder, f"best_model_{model_id}.pt")
    if not os.path.exists(weights_path): continue

    try:
        model = load_model_arch(model_id)
        ckpt = torch.load(weights_path, map_location=DEVICE)
        # Δοκιμή και των δύο πιθανών κλειδιών για τα βάρη
        state_dict = ckpt['model_state'] if 'model_state' in ckpt else ckpt['model_state_dict']
        model.load_state_dict(state_dict, strict=False)
        model.eval()
    except Exception as e:
        print(f"❌ Error loading {model_name}: {e}")
        continue

    print(f"\n🖥️ Displaying results for Model: {model_name}")

    with torch.no_grad():
        for idx in final_target_indices:
            batch_data = VIS_LOADER.dataset[idx]
            inputs = batch_data["image"].unsqueeze(0).to(DEVICE)
            labels = batch_data["label"].cpu().numpy()[0]

            with torch.amp.autocast('cuda'):
                outputs = sliding_window_inference(inputs, ROI_SIZE, 2, model, overlap=0.5)
                pred = (torch.sigmoid(outputs) > 0.5).cpu().numpy()[0, 0]

            # Επιλογή τομής με τον μέγιστο όγκο (ground truth)
            max_slice_idx = np.argmax(np.sum(labels, axis=(0, 1)))

            # Plotting
            img_slice = np.rot90(inputs[0, 0, :, :, max_slice_idx].cpu().numpy())
            lbl_slice = np.rot90(labels[:, :, max_slice_idx])
            pred_slice = np.rot90(pred[:, :, max_slice_idx])

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))

            # Column 1: MRI
            axes[0].imshow(img_slice, cmap="gray")
            axes[0].set_title(f"Idx {idx} | MRI T1c")
            axes[0].axis("off")

            # Column 2: Ground Truth
            axes[1].imshow(img_slice, cmap="gray")
            masked_lbl = np.ma.masked_where(lbl_slice == 0, lbl_slice)
            axes[1].imshow(masked_lbl, cmap="spring", alpha=0.6)
            axes[1].set_title("Ground Truth")
            axes[1].axis("off")

            # Column 3: Model Prediction
            axes[2].imshow(img_slice, cmap="gray")
            masked_pred = np.ma.masked_where(pred_slice == 0, pred_slice)
            axes[2].imshow(masked_pred, cmap="autumn", alpha=0.6)
            axes[2].set_title(f"{model_name} Prediction")
            axes[2].axis("off")

            plt.show() # Απευθείας προβολή

    del model
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import numpy as np
import os

# --- Manually Extracted Data from Logs (Epochs 1-12) ---
# Assuming Validation ran EVERY epoch (VAL_INTERVAL = 1) based on logs
history_data = {
    "epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "train_loss": [
        1.1278, 0.9290, 0.8448, 0.7813, 0.7368, 0.6893,
        0.6445, 0.6134, 0.6050, 0.5952, 0.5883, 0.5871
    ],
    # Validation loss was not printed in the logs, represented as None/NaN
    "val_loss": [
        None, None, None, None, None, None,
        None, None, None, None, None, None
        # Replace None with actual values if you find them logged elsewhere
    ],
    # Validation Dice score logged every epoch
    "val_dice": [
        0.3291, 0.6399, 0.7153, 0.6953, 0.6740, 0.7350,
        0.7006, 0.7078, 0.7099, 0.7069, 0.7392, 0.7392
    ]
}
# --- End Extracted Data ---

# Define the output directory and filename
output_dir = "/content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom"
history_csv_path = os.path.join(output_dir, "training_history_epochs_1_12_corrected.csv") # New filename

# Create DataFrame
try:
    df_history = pd.DataFrame(history_data)

    # Save to CSV
    os.makedirs(output_dir, exist_ok=True) # Ensure directory exists
    df_history.to_csv(history_csv_path, index=False)
    print(f"Successfully created corrected training history CSV: {history_csv_path}")

    # Display the DataFrame (optional)
    print("\nCorrected History DataFrame:")
    from IPython.display import display
    display(df_history)

except Exception as e:
    print(f"Error creating or saving CSV: {e}")